# Encoder swaps in a fixed BioLangFusion pipeline

This notebook asks whether changing one pretrained sequence encoder improves prediction of a measured biological outcome, and whether combining its representation with the other two streams adds value beyond using a single encoder. Those are separate questions: a stronger replacement can improve the whole model even when fusion contributes no further gain.

Stage 1 compared information accessible in frozen encoder representations and similarities between them. [Phase A](../stage3/blf_reference_benchmarks.ipynb) established the original BioLangFusion comparison. Phase B keeps that local prediction pipeline and evaluation in place while replacing one encoder at a time. It examines H2, the project hypothesis linking representation similarity with useful complementarity; similarity alone does not establish that fusion will help.

The reading route follows that investigation: identify the inputs and replacements, prepare or reuse token embeddings, fit the single-encoder and fusion comparisons, then distinguish score improvements from additional fusion benefit. Comparing heads on the same encoder triple adds a third question: does the way representations are combined improve prediction over a simpler combination? RNA-FM versus mRNA-FM provides a worked example. These comparisons characterize the retained tasks; a rule for choosing fusion would also need a stated expectation tested on a new comparison.

**Project context:** research plan item 1, Phase B; [Gantt and Details task 24](../../docs/Research_Project_Plan_Gantt.xlsx); Overleaf Section 12.2, item 1. The [Stage 4 overview](README.md#phase-b-changing-one-encoder-at-a-time) connects this comparison to the surrounding experiments.

## Inputs and the biological questions

Each benchmark row pairs a supplied nucleotide sequence with a target and a training, validation, or test assignment. The targets describe different assays; the shared `Value` column does not give them common biological units.

| Benchmark | Target being predicted | Meaning of a better prediction |
| --- | --- | --- |
| [CoV-Vac](../../docs/methods/data-and-targets.md#cov-vac-chemical-degradation-of-designed-rna) | Standardized chemical degradation of designed RNA | Better ordering of the supplied degradation scores; higher labels mean greater degradation |
| [Fungal expression](../../docs/methods/data-and-targets.md#fungal-expression) | Log RNA-seq expression | Better ordering of RNA abundance, not directly measured protein production |
| [E. coli expression](../../docs/methods/data-and-targets.md#protein-expression-in-e-coli) | Low, medium, or high protein-expression class from gel scoring | More correct expression-category predictions in the E. coli host |
| [mRNA stability](../../docs/methods/data-and-targets.md#mrna-stability-standardized-decay-profiles) | Decay slope standardized within its experimental profile and oriented toward stability | Better ordering of relative stability scores, not prediction of lifetime in hours |

The loader retains the supplied split assignments and drops sequences longer than 999 nucleotides; it does not truncate them to 999. This inherited choice reproduces the local BioLangFusion comparison rather than establishing an optimal biological sequence length. See the [Stage 3 dataset audit](../stage3/blf_datasets.ipynb) and [preprocessing and evaluation limits](../../docs/methods/data-and-targets.md#splits-preprocessing-and-the-claim-being-tested).

### From one supplied sequence to three encoder streams

DNA and RNA encoders read alphabet-specific versions of the same nucleotide string, including T-to-U conversion for the RNA view. The protein stream is computed by translating complete triplets from the first supplied base, representing stop codons as `X` to retain the positional grid. It is not an independently measured protein sequence. This matters especially for CoV-Vac: its RNA fragments are not verified coding sequences, so the computed protein input should not be described as their demonstrated translation product. [Input construction](../shared_code/fusion/token_embeddings.py)

An encoder writes a numerical feature vector at each token position. Tokens may represent one nucleotide, a three-base codon, a six-base segment, or one amino-acid residue. These token embeddings retain positions, unlike the one-vector-per-sequence pooled embeddings used in the mRFP ridge analysis. A trainable TextCNN prediction head scans neighboring token vectors to predict the target; its kernel size is the number of neighboring positions each filter reads.

The **codon grid** puts the streams on corresponding triplet positions before fusion. Nucleotide-level states are averaged in threes; six-base DNA states use a learned upsampling operation; codon and residue states already have the required spacing. This is computational correspondence under the supplied frame, not proof of biological translation. The replacement table makes each conversion explicit.

### What changes and what stays fixed

Each comparison replaces the encoder in one input slot and retains the other two encoders. It preserves the benchmark rows, split assignments, grid method, prediction-head recipe, and evaluation. Encoder width and token resolution can still change, so this is not an isolation of pretraining data alone.

| Slot | Original encoder | Replacement | Placement on the codon grid |
| --- | --- | --- | --- |
| DNA | Nucleotide Transformer v2 100M | NT 500M human-ref | 6-mers, using the same transposed convolution |
| DNA | Nucleotide Transformer v2 100M | HyenaDNA large | Nucleotide tokens averaged in threes |
| RNA | RNA-FM | RiNALMo | Nucleotide tokens averaged in threes |
| RNA | RNA-FM | mRNA-FM, CaLM | Codon tokens used directly |
| Protein | ESM-2 8M | ESM-2 35M, ESM-2 150M, ProtBERT | Residue tokens used directly |

**Why these comparisons.** The plan names CaLM, mRNA-FM, ProtBERT, ESM-2 150M and alternative DNA encoders. RiNALMo and ESM-2 35M extend the comparison to encoders that Stage 1 found similar to those they replace. Under H2, that similarity motivates testing whether they add less useful information; it does not establish the outcome. DNABERT-2 remains outside this sweep because its variable-length byte-pair tokens need a separately designed, fixed grid adapter.

## RNA-FM and mRNA-FM as a worked encoder comparison

This replacement helps connect the biological inputs, pretrained models and fusion results. Both encoders receive the same supplied nucleotide sequence in the RNA alphabet. The question is whether their different representations change prediction, and whether the DNA and protein streams then add value beyond the replacement alone. The interpretation below was developed from the retained results; it does not reconstruct every historical reason for choosing the models.

### 1. What each encoder learned from

Noncoding RNAs function without being translated into proteins. A coding sequence (CDS) is the part of an mRNA read in three-nucleotide codons during translation; it excludes the untranslated regions. These are different pretraining populations, although both contain nucleotide sequence patterns.

| Model property | RNA-FM | mRNA-FM |
| --- | --- | --- |
| Reported pretraining population | 23.7 million noncoding RNA sequences from RNAcentral, after removing identical sequences | 45 million coding sequences |
| Token unit | One nucleotide | One nonoverlapping three-nucleotide codon |
| Transformer blocks | 12 | 12 |
| Features per content token | 640 | 1,280 |
| Approximate parameter count | 99 million | 239 million |

The [RNA-FM paper's Methods](https://arxiv.org/html/2204.00300v5) describe masked-nucleotide prediction: recover hidden bases from surrounding sequence. The [authors' model documentation](https://github.com/ml4bio/RNA-FM#foundation-models) identifies the mRNA-FM corpus and codon representation. The inspected author materials do not establish mRNA-FM's exact masking recipe, source-database mixture or organism proportions; RNA-FM's detailed training procedure should not simply be assigned to it.

**What changes in this notebook.** RNA-FM represents individual bases, while mRNA-FM represents triplets aligned from the start of the supplied sequence. Both can distinguish synonymous codons; codon tokens impose that grouping rather than introduce sequence information absent from RNA-FM. For fusion, three consecutive RNA-FM states are averaged onto one grid position, while mRNA-FM already supplies one state per codon. With the 999-nucleotide filter and configured 1,024-token limits, both cover the retained sequence; extra sequence coverage is not the explanation for a difference here.

The replacement also changes encoder capacity and predictor inputs. Native TextCNN kernels of three, four and five tokens directly read 3–5 nucleotide positions with RNA-FM but 9–15 with mRNA-FM. These spans describe the token positions read by the head; each contextual encoder vector can already reflect a broader sequence. The wider mRNA-FM vectors also increase trainable input-layer dimensions. The separate RNA-FM codon-grid control helps examine positional resolution, but does not isolate the pretraining corpus or model capacity. See the [input conversion and grid construction](../shared_code/fusion/token_embeddings.py) and [prediction head](../shared_code/fusion/textcnn.py).

### 2. What capabilities are supported by evidence?

The RNA-FM paper evaluates task-specific predictors using its representations for RNA structure and interactions, and for mean ribosome load from mRNA 5′ untranslated regions. That last result matters here: noncoding-RNA pretraining does not make RNA-FM inherently unsuitable for an mRNA-related task. These evaluations concern the paper's predictors and datasets, not direct measurements of structure in our benchmark rows. [RNA-FM Results, Figure 6 and Table 5](https://arxiv.org/html/2204.00300v5)

Those results belong to RNA-FM. The inspected paper does not supply an equivalent mRNA-FM evaluation, and the author release's general performance description does not establish a quantitative advantage on our targets. Our [mRFP control](../stage1/mrfp_representation_control.ipynb) supplies separate local evidence: mRNA-FM's pooled representations improve prediction of synonymous variants' expression beyond the selected linear composition and folding controls. That uses ridge regression and one reporter protein; the token-level comparisons here test transfer to other assays and a different predictor.

The biological expectation is therefore a hypothesis: a representation learned from CDS may make useful coding-sequence patterns easier for the predictor to use. The settings below retain the experiment that tests this expectation. The result discussion returns to where it holds, where it fails, and what remains unresolved.

---

## Setup

The notebook supports NVIDIA CUDA, Apple MPS, and CPU execution. The device cell reports the selected backend before extraction or fitting. No hardware-specific runtime or peak RAM/VRAM requirement is established here; token-array disk estimates below are not memory requirements or duration forecasts.

### Imports

Use the environment described in the [repository setup instructions](../../README.md#setup). Dependency roles are noted beside their imports; the following cells locate shared project code and report the compute device.

In [1]:
# Files, environment, and elapsed time
import json
import os
import sys
import time
from pathlib import Path

# Set Apple GPU fallback before importing PyTorch; preserve an explicit user setting.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# Numerical arrays and result tables
import numpy as np
import pandas as pd

# Tensor operations, model training, and available compute backends
import torch

# Render notebook tables explicitly rather than relying on an implicit kernel name.
from IPython.display import display

### Project paths and shared code

Locate the repository's `Code` directory so the same shared functions can be imported from the repository root or a notebook folder.

In [2]:
# Search the working directory and its parents for Code/shared_code.
CODE_DIR = next(
    (
        candidate.resolve()
        for parent in (Path.cwd(), *Path.cwd().parents)
        for candidate in (parent, parent / "Code")
        if (candidate / "shared_code").is_dir()
    ),
    None,
)
if CODE_DIR is None:
    raise RuntimeError(
        "Run this notebook from the repository root or a directory inside it "
        "so Code/shared_code can be found."
    )
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Benchmark rows and encoder definitions
from shared_code import datasets
from shared_code.encoders import ENCODERS

# Fusion input streams, prediction-head training, and token-cache verification
from shared_code.fusion import blf, train, swap_results, swap_plan, run_identity
from shared_code.fusion import token_embeddings as tokens

### Compute device

Inspect the device report before extraction or training. Availability depends on the installed PyTorch build and local hardware; this notebook supports Apple MPS, NVIDIA CUDA, and CPU. The report identifies the selected backend, not a runtime or memory estimate.

In [3]:
# Table display width (characters) and visible columns, not sequence or token limits.
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

# Preserve the existing preference: Apple MPS, then NVIDIA CUDA, then CPU.
DEVICE = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(
    "torch", torch.__version__,
    "| CUDA available:", torch.cuda.is_available(),
    "| Apple MPS available:", torch.backends.mps.is_available(),
)
if DEVICE == "cuda":
    print(f"Selected device: {DEVICE} | GPU: {torch.cuda.get_device_name()}")
elif DEVICE == "mps":
    print(f"Selected device: {DEVICE} | GPU: Apple GPU (MPS)")
else:
    print(f"Selected device: {DEVICE}")

torch 2.10.0+cu126 | CUDA available: True | Apple MPS available: False
Selected device: cuda | GPU: NVIDIA GeForce RTX 2060 SUPER


### Settings and comparison scope

**Benchmarks.** The configured sweep covers CoV-Vac, fungal expression, E. coli and mRNA stability. Each benchmark includes all eight configured replacements. The earlier stability replacement subset included only mRNA-FM, CaLM and ESM-2 150M, which explains the narrower subset in some retained outputs. Use the fit-coverage and execution-plan tables below to establish what is available for the selected seed; configuration counts and earlier interval tables do not establish current completion. This is the expanded comparison described by task 24 in the [research plan](../../docs/Research_Project_Plan_Gantt.xlsx). Ab1 remains excluded because its data are not public.

**Manual seeds.** Leave `READ_SAVED_RESULTS = True` for inspection. To complete the configured fits, set it to `False` and begin with `SEED = 0`. After that run finishes, manually change `SEED` to `1`, run from the top, then do the same for `2`. The notebook never advances the seed automatically. It reuses compatible completed records for the requested seed and trains only missing entries. Frozen token embeddings are shared across seeds; changing the training seed does not require extracting them again.

**Fixed selection.** Keep `SELECTION_SEED = 0`. The original encoders' complete positive-penalty validation grid selects the gated-head penalty at that seed. Each triple's best-single baseline is also chosen using its seed-0 validation scores, then that same model variant is compared at the requested training seed. Neither choice is retuned for seeds 1 or 2. Repeat preflight therefore requires the seed-0 single-encoder choices to be available before new work starts.

**Where work is saved.** Token caches remain under `Code/stage3/embeddings/<benchmark>/<encoder>/`. Replacement fits use `Code/stage4/results/phaseB/<benchmark>/<encoder>/`. Missing original-encoder references use `Code/stage4/results/phaseA/<benchmark>/`; compatible earlier originals may also be read from Stage 3. Filenames retain the actual training seed. No earlier result is renamed or replaced to fill a missing prediction or metadata field.

**Reading and analysis.** Default Run All reads result evidence and reports missing work without loading datasets or token arrays, fitting, or writing results. Historical identity limits remain explicit. In computation mode, `RUN_BOOTSTRAP = False` keeps interval resampling separate from completing fits, so a repeated pass does not recalculate that analysis. The historical interval calculation requires both `READ_SAVED_RESULTS = False` and `RUN_BOOTSTRAP = True`. It does not fit models itself, but earlier cells can process pending fits in that computation mode. The separate saved-comparison report below uses `READ_SAVED_RESULTS = True` to keep fitting disabled.


In [4]:
# Default Run All reads saved evidence; changing this enables research computation.
READ_SAVED_RESULTS = True

BENCHMARKS = (
    "cov_vac_degradation",
    "fungal_expression",
    "ecoli_proteins",
    "mrna_stability",
)
CACHE_ROOT = CODE_DIR / "stage3" / "embeddings"
RESULTS_ROOT = CODE_DIR / "stage4" / "results" / "phaseB"
COMPARISON_DIR = RESULTS_ROOT / "analyses" / "paired-report-v1"

# Original-encoder results are read for comparison and validation-selected MIL weights.
ORIGINAL_RESULTS = {
    "cov_vac_degradation": [Path("stage4/results/phaseA/cov_vac_degradation")],
    "ecoli_proteins": [Path("stage4/results/phaseA/ecoli_proteins")],
    "fungal_expression": [
        Path("stage3/results/fungal_expression"),
        Path("stage4/results/phaseA/fungal_expression"),
    ],
    "mrna_stability": [Path("stage4/results/phaseA/mrna_stability")],
}
ORIGINAL_RESULTS = {
    benchmark: [CODE_DIR / folder for folder in folders]
    for benchmark, folders in ORIGINAL_RESULTS.items()
}
STAGE1_ALIGNMENT = CODE_DIR.parent / "docs" / "figures" / "stability" / "results.json"

#### Training settings and replacement choices

The pretrained encoders stay frozen. A trainable **TextCNN** scans local windows of token features and summarizes them to predict the target. Here it retains 100 learned filters for each kernel size of three, four, and five positions; the biological span of a position depends on the encoder or grid. The [prediction-head implementation](../shared_code/fusion/textcnn.py) records the shared architecture and the paper's width ambiguity.

`MAX_EPOCHS` is an upper limit on passes through training data, not a promised duration. The shared [training configuration](../shared_code/fusion/train.py) retains validation-based early stopping. `DTYPE` controls stored token arrays; it does not select half-precision encoder loading or training.

In [5]:
# Prediction-head training settings
TCONV = "equation"  # Retain the Equation 1 six-mer-to-codon upsampling.
CHANNELS = 100      # Learned filters per TextCNN kernel size, not encoder width.
BATCH_SIZE = 32     # Benchmark rows per training minibatch.
MAX_EPOCHS = 500    # Ceiling; validation-based early stopping may end a fit sooner.
SEED = 0           # Change manually to 1, then 2; one training seed per execution.
SELECTION_SEED = 0  # Keep validation-selected penalties and baseline choices fixed.
RUN_BOOTSTRAP = False  # Separate analysis; leave off when completing fits or repeats.

# Independent CPU analysis of saved predictions; no fitting or embedding extraction.
RUN_SAVED_COMPARISONS = False
COMPARISON_SEEDS = (0, 1, 2)  # Report each available seed; mean requires this entire set.
COMPARISON_DRAWS = 2000
COMPARISON_BOOTSTRAP_SEED = 0
if RUN_SAVED_COMPARISONS and not READ_SAVED_RESULTS:
    raise ValueError(
        "Set READ_SAVED_RESULTS = True for saved-prediction reporting; "
        "training and extraction must remain disabled."
    )

# Cache chunking and stored-array precision, not encoder or training precision
CHUNK_SIZE = tokens.DEFAULT_CHUNK
DTYPE = np.float16

# Each pair replaces one slot while retaining the other two original encoders.
SWAPS = [
    ("dna", "nt500m"),
    ("dna", "hyenadna"),
    ("rna", "rinalmo"),
    ("rna", "mrnafm"),
    ("rna", "calm"),
    ("prot", "esm2_35m"),
    ("prot", "esm2_150m"),
    ("prot", "protbert"),
]
# Include all eight replacements; completed fits in the former subset are reused.
STABILITY_SWAPS = list(SWAPS)
ORIGINAL = dict(blf.ENCODER_OF_MODALITY)

#### Select the comparisons and shared caches

`swaps_for` returns the configured replacement list for each benchmark. Stability now uses the same eight replacements as the other benchmarks. `required_encoders` lists the original streams and replacements without duplicates. The execution plan below narrows cache work to encoders needed by genuinely pending fits; completed comparisons do not require their large arrays to be opened.


In [6]:
def swaps_for(benchmark):
    """Use the configured stability replacements or the full swap list for other benchmarks."""
    return STABILITY_SWAPS if benchmark == "mrna_stability" else SWAPS


def required_encoders(benchmark):
    """List the original and replacement encoders needed for one benchmark."""
    # Remove duplicates without changing order: original streams are checked first.
    return list(dict.fromkeys([
        *ORIGINAL.values(),
        *(encoder_key for _, encoder_key in swaps_for(benchmark)),
    ]))

#### Plan the selected seed

The plan distinguishes reusable completed records, new fits, resumable checkpoints and blocked evidence. It includes original references and replacements separately, then lists entries needing attention. In computation mode every benchmark must pass before any extraction begins. When a checkpoint needs inspection, its recorded recipe and recovery state are checked on CPU; read-only mode does not deserialize it. A plan is a snapshot, not proof of unrecorded historical provenance.

The definitions below describe that procedure; the following cell builds and displays the plan for the selected seed.


In [ ]:
def result_recipe(seed):
    """Match the full retained TrainConfig, including defaults not exposed above."""
    return vars(train.TrainConfig(
        batch_size=BATCH_SIZE,
        max_epochs=MAX_EPOCHS,
        channels=CHANNELS,
        tconv=TCONV,
        seed=seed,
    ))


# Plan every requested fit before extraction. Reuse actual saved paths, including
# compatible historical names; a missing result is different from a conflicting file.
def requested_identity(benchmark, variant, config, mapping, *, rows=None, verification=None):
    """Bind the fit to ordered rows and cache metadata without opening token arrays."""
    if rows is None:
        rows = datasets.blf_benchmark_rows(benchmark)
    spec = datasets.DATASETS[benchmark]
    records = {
        modality: run_identity.cache_record(
            CACHE_ROOT / benchmark / encoder_key, verification=verification,
        )
        for modality, encoder_key in mapping.items()
    }
    return run_identity.blf_identity(
        rows=rows, benchmark=benchmark, variant=variant, config=config,
        encoders=mapping, caches=records, task=spec.task,
        label_column=spec.label_column, split_column=spec.split_column,
    )


def execution_plan(benchmark):
    """Use the same preflight before extraction and immediately before fitting."""
    return swap_plan.plan_benchmark(
        benchmark=benchmark,
        config=result_recipe(SEED),
        selection_seed=SELECTION_SEED,
        original_encoders=ORIGINAL,
        original_folders=ORIGINAL_RESULTS[benchmark],
        original_destination=CODE_DIR / "stage4" / "results" / "phaseA" / benchmark,
        swaps=[
            (modality, encoder_key, tokens.grid_resolution(encoder_key) != "codon")
            for modality, encoder_key in swaps_for(benchmark)
        ],
        swap_root=RESULTS_ROOT / benchmark,
        checkpoint_validator=None if READ_SAVED_RESULTS else train.checkpoint_preflight,
        identity_factory=None if READ_SAVED_RESULTS else requested_identity,
    )


In [7]:
FIT_PLANS = {benchmark: execution_plan(benchmark) for benchmark in BENCHMARKS}
fit_inventory = pd.DataFrame([row for plan in FIT_PLANS.values() for row in plan.rows()])
if not fit_inventory.empty:
    display(fit_inventory.groupby(["benchmark", "role", "status"]).size().rename("fits"))
    display(fit_inventory.loc[fit_inventory["status"] != "reuse"])
for benchmark, plan in FIT_PLANS.items():
    for issue in plan.issues:
        print(f"{benchmark}: {issue}")

if not READ_SAVED_RESULTS:
    # Stop the entire request before expensive work if any comparison is blocked.
    for plan in FIT_PLANS.values():
        plan.require_ready()

PENDING_RUNS = [
    run for plan in FIT_PLANS.values() for run in plan.runs
    if run.status in ("new", "resume")
]
PENDING_CACHE_KEYS = {
    (run.benchmark, encoder_key)
    for run in PENDING_RUNS for encoder_key in run.encoders.values()
}
print(f"Training seed {SEED}; selection seed {SELECTION_SEED}; "
      f"{len(PENDING_RUNS)} new/resumable fits in the displayed plan.")


benchmark            role         status
cov_vac_degradation  original     reuse      7
                     replacement  reuse     35
ecoli_proteins       original     reuse      7
                     replacement  reuse     35
fungal_expression    original     reuse      7
                     replacement  reuse     35
mrna_stability       original     reuse      7
                     replacement  reuse     35
Name: fits, dtype: int64

,benchmark,role,encoder,variant,seed,lambda,status,reason,source,destination,legacy identity


Training seed 0; selection seed 0; 0 new/resumable fits in the displayed plan.


In [8]:
planned = sum(len(swaps_for(benchmark)) for benchmark in BENCHMARKS)
print(f"{planned} swap-benchmark pairs over {len(BENCHMARKS)} benchmarks, device {DEVICE}")
pd.DataFrame([
    {
        "slot": modality,
        "replaces": ENCODERS[ORIGINAL[modality]].label,
        "swapped in": ENCODERS[encoder_key].label,
        "grid": tokens.grid_resolution(encoder_key),
    }
    for modality, encoder_key in SWAPS
])


32 swap-benchmark pairs over 4 benchmarks, device cuda


,slot,replaces,swapped in,grid
0,dna,NT v2 100M multi-species,NT 500M human-ref,kmer
1,dna,NT v2 100M multi-species,HyenaDNA large,nucleotide
2,rna,RNA-FM,RiNALMo 150M,nucleotide
3,rna,RNA-FM,mRNA-FM,codon
4,rna,RNA-FM,CaLM,codon
5,prot,ESM-2 8M,ESM-2 35M,codon
6,prot,ESM-2 8M,ESM-2 150M,codon
7,prot,ESM-2 8M,ProtBERT,codon


### Estimate token-array storage

This computation-only projection reads rows for benchmarks with pending fits and estimates the selected replacement arrays' payload from token counts, stored dtype and encoder width. It does not extract embeddings. It can include already cached arrays needed by a missing fit, so the total is not additional free space required. Full cache checks below establish which shards actually need extraction. Model downloads, temporary publication space and peak RAM/GPU allocations are not measured by this projection.


In [ ]:
# Token-array payload only; widths describe the selected checkpoints.
WIDTHS = {
    "nt500m": 1280,
    "hyenadna": 256,
    "rinalmo": 640,
    "mrnafm": 1280,
    "calm": 768,
    "esm2_35m": 480,
    "esm2_150m": 640,
    "protbert": 1024,
}


def estimate_cache_storage(benchmarks, pending_cache_keys, encoder_widths, dtype):
    """Estimate full token-array sizes for replacement caches used by pending fits."""
    pending_benchmarks = {
        benchmark for benchmark, _ in pending_cache_keys
    }
    bytes_per_feature = np.dtype(dtype).itemsize
    storage_rows = []

    for benchmark in benchmarks:
        if benchmark not in pending_benchmarks:
            continue

        sequence_lengths = datasets.blf_benchmark_rows(benchmark)["length"].to_numpy()

        for _, encoder_key in swaps_for(benchmark):
            if (benchmark, encoder_key) not in pending_cache_keys:
                continue

            encoder = ENCODERS[encoder_key]
            token_counts = np.array([
                tokens.tokens_for_length(encoder, int(length))
                for length in sequence_lengths
            ])
            # Content tokens x features/token x bytes/feature; decimal GB.
            # This excludes model and training memory.
            gigabytes = (
                token_counts.sum()
                * encoder_widths[encoder_key]
                * bytes_per_feature
                / 1e9
            )
            storage_rows.append({
                "benchmark": benchmark,
                "encoder": encoder.label,
                "rows": len(sequence_lengths),
                "GB": round(gigabytes, 1),
            })

    return pd.DataFrame(
        storage_rows,
        columns=["benchmark", "encoder", "rows", "GB"],
    )


In [9]:
projection = pd.DataFrame()

if READ_SAVED_RESULTS:
    print("Storage projection skipped: reading saved results does not load benchmark inputs.")
elif not PENDING_RUNS:
    print("All requested fits are complete; no extraction storage is needed.")
else:
    projection = estimate_cache_storage(
        BENCHMARKS,
        PENDING_CACHE_KEYS,
        WIDTHS,
        DTYPE,
    )
    display(
        projection.pivot_table(
            index="benchmark",
            columns="encoder",
            values="GB",
            aggfunc="sum",
        ).fillna("")
    )
    print(
        "Estimated replacement-cache payload needed by pending fits, including existing caches: "
        f"~{projection['GB'].sum():.0f} GB"
    )
    print("All of it stays out of Git, under Code/stage3/embeddings/<benchmark>/<encoder>/.")


Storage projection skipped: reading saved results does not load benchmark inputs.


**Reading the storage projection.** The table above multiplies expected content-token counts by a hard-coded hidden width and the cache dtype size. Its decimal GB values cover token-array payload only, excluding model downloads, archive and metadata overhead, training checkpoints and other saved weights. They do not establish which resource will limit a run.

**Width and saved-output boundary.** The projection now uses 640 features per RiNALMo token, matching the inspected token-array headers in the current CoV-Vac, fungal and E. coli caches. The preserved output was produced with the earlier 1,280-width lookup and can therefore show twice the RiNALMo payload. This correction affects a storage estimate, not embeddings or fitted results. Token payload alone remains insufficient to estimate training RAM, VRAM or runtime.

### Check the original encoders' results

Phase B needs these runs both as score references and to select the gated head's entropy weight: Phase A for CoV-Vac, E. coli and mRNA stability; Stage 3 step 3 or the configured Phase A folder for fungal. Check the table below for available variants and a gated lambda before starting training. The fit plan above uses those configured paths to find compatible records. It requires the completed seed-0 tuning grid before work; when repeating a training seed, it schedules that seed's missing original references before replacements.

In [10]:
def result_variant_label(result):
    """Keep unregularized MIL separate from the selected positive-penalty variant."""
    return (
        "mil (lam 0)"
        if result.variant == "mil" and result.config["entropy_lambda"] == 0
        else result.variant
    )


def original_report(benchmark, seed):
    return swap_results.read_results(
        ORIGINAL_RESULTS[benchmark],
        benchmark=benchmark,
        config=result_recipe(seed),
        encoders=ORIGINAL,
        native_fallback=True,
    )


def original_runs(benchmark, seed=None):
    """Read the selected seed; ambiguous identities cannot silently replace one another."""
    report = original_report(benchmark, SEED if seed is None else seed)
    # Duplicated identities are withheld by the reader; other saved scores still stand.
    records = report.records if READ_SAVED_RESULTS else report.require_unique()
    return [train.Result.load(record["path"]) for record in records]


def selected_mil_penalty(benchmark):
    """Reuse the complete original positive-penalty grid's seed-0 selection."""
    mil_penalty = FIT_PLANS[benchmark].lambda_
    if mil_penalty is None:
        raise RuntimeError(f"{benchmark}: seed-0 MIL selection unavailable; inspect the execution plan")
    return mil_penalty


In [ ]:
def swap_report(benchmark, modality, encoder_key, seed, mil_penalty):
    """Check the replacement recipe, encoder mapping when recorded, and fit names."""
    expected = swap_results.swap_variants(
        modality, grid=tokens.grid_resolution(encoder_key) != "codon",
    )
    return swap_results.read_results(
        [RESULTS_ROOT / benchmark / encoder_key],
        benchmark=benchmark,
        config=result_recipe(seed),
        encoders={**ORIGINAL, modality: encoder_key},
        mil_lambda=mil_penalty,
        variants=(
            expected if mil_penalty is not None
            else tuple(variant for variant in expected if variant != "mil")
        ),
    )

The next cell reads the configured result folders and lists the available variants. A missing gated lambda indicates an unmet prerequisite; this check does not fit a replacement model.

In [11]:
status = []
for benchmark in BENCHMARKS:
    report = original_report(benchmark, SEED)
    available_variants = sorted({
        result_variant_label(train.Result.load(record["path"]))
        for record in report.records
    })
    try:
        mil_penalty = selected_mil_penalty(benchmark)
    except RuntimeError as exc:
        # Display the prerequisite failure before any extraction or fitting.
        mil_penalty = None
        prerequisite = str(exc)
    else:
        prerequisite = ""
    status.append({
        "benchmark": benchmark,
        "original runs": len(report.records),
        "variants": ", ".join(available_variants),
        "gated lambda": mil_penalty,
        "prerequisite issue": prerequisite,
        "excluded files": len(report.issues),
    })
pd.DataFrame(status).set_index("benchmark")

,original runs,variants,gated lambda,prerequisite issue,excluded files
benchmark,,,,,
cov_vac_degradation,10,"concat, cross, mil, mil (lam 0), unimodal-grid...",0.01,,8
fungal_expression,10,"concat, cross, mil, mil (lam 0), unimodal-grid...",0.01,,16
ecoli_proteins,10,"concat, cross, mil, mil (lam 0), unimodal-grid...",0.01,,8
mrna_stability,10,"concat, cross, mil, mil (lam 0), unimodal-grid...",0.01,,0


### Check saved-result coverage before computation

This read-only check separates saved fits from token-cache readiness. It checks the direct result folders for the declared training recipe, identifies excluded or ambiguous records, and shows which comparisons have saved scores and paired prediction arrays. It does not open token caches, load encoders, fit a model, or calculate new scores.

Seeds 0, 1 and 2 are inspected to make missing repeats visible. This inspection does not enable those repeats: fitting and the comparison tables still use `SEED`. Coverage for every inspected seed uses the positive MIL penalty selected by validation at `SELECTION_SEED = 0`; it does not retune the penalty for a future repeat. A replacement also needs the unchanged single-encoder references and original fusion heads from the same inspected seed. This coverage check only reports missing references; the separate execution plan schedules them when computation is enabled.

“Legacy identity” counts eligible records whose encoder mapping was not recorded. Folder placement is the historical attribution for these files, not a new verification of their inputs or producer. “Without paired arrays” distinguishes an available score from evidence usable for the later bootstrap. The detailed exclusions remain in `result_issues`; the displayed reason counts explain why files were not selected. The check is a snapshot when its display cell is run, not a monitor of another process.

Coverage also lists the original RNA-FM grid control when RNA is unchanged, and flags score-only reference records separately. The score reader retains the historical rule that original grid controls are eligible where available; the execution plan includes a missing same-seed control needed to retain that candidate set.

In [ ]:
def coverage_for_benchmark(benchmark, seeds):
    """Collect saved-fit coverage and exclusions for one benchmark and its references."""
    coverage_rows, issue_rows = [], []
    original_variants = swap_plan.ORIGINAL_VARIANTS

    try:
        inherited_lambda = selected_mil_penalty(benchmark)
        prerequisite = ""
    except RuntimeError as exc:
        inherited_lambda = None
        prerequisite = str(exc)

    # Zero here marks unavailable positive-penalty selection; it does not select MIL(0).
    coverage_lambda = inherited_lambda if inherited_lambda is not None else 0.0
    for inspected_seed in seeds:
        originals = original_report(benchmark, inspected_seed)
        # The coverage row concerns the inherited penalty, not another seed's tuning winner.
        original_coverage = originals.coverage(original_variants, mil_lambda=coverage_lambda)
        coverage_rows.append({
            "benchmark": benchmark, "encoder": "original", "seed": inspected_seed,
            **original_coverage, "reference gaps": "", "reference arrays absent": "",
            "prerequisite": prerequisite,
        })
        issue_rows.extend({
            "benchmark": benchmark, "encoder": "original", "seed": inspected_seed, **issue,
        } for issue in originals.issues)

        for modality, encoder_key in swaps_for(benchmark):
            expected = swap_results.swap_variants(
                modality, grid=tokens.grid_resolution(encoder_key) != "codon",
            )
            report = swap_report(
                benchmark, modality, encoder_key, inspected_seed, inherited_lambda,
            )
            references = [
                *(
                    f"unimodal:{reference_modality}"
                    for reference_modality in ORIGINAL if reference_modality != modality
                ),
                *(("unimodal-grid:rna",) if modality != "rna" else ()),
                "concat", "mil", "cross",
            ]
            reference_coverage = originals.coverage(references, mil_lambda=coverage_lambda)
            coverage_rows.append({
                "benchmark": benchmark, "encoder": ENCODERS[encoder_key].label,
                "seed": inspected_seed,
                **report.coverage(expected, mil_lambda=coverage_lambda),
                "reference gaps": reference_coverage["missing"],
                "reference arrays absent": reference_coverage["without paired arrays"],
                "prerequisite": prerequisite,
            })
            issue_rows.extend({
                "benchmark": benchmark, "encoder": encoder_key, "seed": inspected_seed, **issue,
            } for issue in report.issues)
    return coverage_rows, issue_rows


In [12]:
# Inspection only: these seeds do not alter SEED or launch additional fits.
COVERAGE_SEEDS = (0, 1, 2)
coverage_rows, issue_rows = [], []
for benchmark in BENCHMARKS:
    benchmark_coverage, benchmark_issues = coverage_for_benchmark(benchmark, COVERAGE_SEEDS)
    coverage_rows.extend(benchmark_coverage)
    issue_rows.extend(benchmark_issues)

result_coverage = pd.DataFrame(coverage_rows).set_index(["benchmark", "encoder", "seed"])
result_issues = pd.DataFrame(issue_rows)
display(result_coverage)
if not result_issues.empty:
    display(result_issues.groupby(["seed", "reason"]).size().rename("excluded records"))

available                            missing without paired arrays ambiguous  excluded files  legacy identity  \
benchmark           encoder           seed                                                                                                                                                          
cov_vac_degradation original          0     unimodal:dna, unimodal:rna, unimodal:prot, uni...                                                                                  8               10   
                    NT 500M human-ref 0     unimodal:dna, unimodal-grid:dna, concat, mil, ...                                                                                  0                5   
                    HyenaDNA large    0     unimodal:dna, unimodal-grid:dna, concat, mil, ...                                                                                  0                5   
                    RiNALMo 150M      0     unimodal:rna, unimodal-grid:rna, concat, mil, ...                                                                                  0                5   
                    mRNA-FM           0                      unimodal:rna, concat, mil, cross                                                                                  0                0   
...                                                                                       ...                                ...                   ...       ...             ...              ...   
mrna_stability      mRNA-FM           2                                                         unimodal:rna, concat, mil, cross                                               4                0   
                    CaLM              2                                                         unimodal:rna, concat, mil, cross                                               4                0   
                    ESM-2 35M         2                                                        unimodal:prot, concat, mil, cross                                               4                0   
                    ESM-2 150M        2                                                        unimodal:prot, concat, mil, cross                                               4                0   
                    ProtBERT          2                                                        unimodal:prot, concat, mil, cross                                               4                0   

                                                                               reference gaps reference arrays absent prerequisite  
benchmark           encoder           seed                                                                                          
cov_vac_degradation original          0                                                                                             
                    NT 500M human-ref 0                                                                                             
                    HyenaDNA large    0                                                                                             
                    RiNALMo 150M      0                                                                                             
                    mRNA-FM           0                                                                                             
...                                                                                       ...                     ...          ...  
mrna_stability      mRNA-FM           2       unimodal:dna, unimodal:prot, concat, mil, cross                                       
                    CaLM              2       unimodal:dna, unimodal:prot, concat, mil, cross                                       
                    ESM-2 35M         2     unimodal:dna, unimodal:rna, unimodal-grid:rna,...                                       
                    ESM-2 150M        2     unimodal:dna, u

seed  reason                                        
0     configuration differs: channels                     1
      configuration differs: seed                        24
      configuration differs: tconv                        6
      native baseline with requested tconv preferred      1
1     configuration differs: channels, seed               1
      configuration differs: seed                       193
      configuration differs: seed, tconv                  6
2     configuration differs: channels, seed               1
      configuration differs: seed                       193
      configuration differs: seed, tconv                  6
Name: excluded records, dtype: int64

---

## Part 1: Prepare the token embeddings

### Check caches before loading models

Inspect the cache pairs required by pending fits, including their original streams. A fully reused fit plan opens no token arrays. The preflight checks extraction settings, recorded ordered-input hashes, shard structure, consistent widths and each row's token count. Available historical shard hashes are also checked. Verification can be reused between preflight and extraction while file statistics remain unchanged; rerunning this preflight creates a fresh verification object.

**Input identity is required.** Reusing a cache requires recorded ordered-input identity; this notebook has no legacy-cache reuse opt-in. New extraction records that identity before the first shard, and a mismatch stops reuse. The [reset record](../../docs/run-records.md#2026-10-25-encoder-swap-unverified-cache-reset) preserves the history of the legacy-cache cleanup.


In read-saved mode, the first cell lists cache metadata without reading shards or establishing compatibility. The following cell performs full preflight only in computation mode, for caches needed by pending fits. Missing metadata does not block the saved-result tables.


In [13]:
def saved_cache_status(benchmark, encoder_key):
    """Read one metadata status row without opening or verifying token shards."""
    path = CACHE_ROOT / benchmark / encoder_key / "meta.json"
    try:
        metadata = json.loads(path.read_text(encoding="utf-8"))
        if not isinstance(metadata, dict):
            raise ValueError("metadata is not an object")
        cache_status = "saved metadata; shards not checked"
        issue = ""
    except (OSError, ValueError) as exc:
        metadata = {}
        cache_status = "metadata unavailable"
        issue = str(exc)
    return {
        "benchmark": benchmark,
        "encoder": ENCODERS[encoder_key].label,
        "status": cache_status,
        "recorded shards": metadata.get("n_shards"),
        "issues": issue,
    }


if READ_SAVED_RESULTS:
    # Inspect small metadata files only. Their presence does not verify the shards.
    CACHE_PLANS = {}
    BENCHMARK_ROWS = {}
    checks = [
        saved_cache_status(benchmark, encoder_key)
        for benchmark in BENCHMARKS
        for encoder_key in required_encoders(benchmark)
    ]
    display(pd.DataFrame(checks).set_index(["benchmark", "encoder"]))
    print("Saved scores remain readable without token caches. No recovery is attempted.")


status  recorded shards issues
benchmark           encoder                                                                             
cov_vac_degradation NT v2 100M multi-species  saved metadata; shards not checked               10       
                    RNA-FM                    saved metadata; shards not checked               10       
                    ESM-2 8M                  saved metadata; shards not checked               10       
                    NT 500M human-ref         saved metadata; shards not checked               10       
                    HyenaDNA large            saved metadata; shards not checked               10       
                    RiNALMo 150M              saved metadata; shards not checked               10       
                    mRNA-FM                   saved metadata; shards not checked               10       
                    CaLM                      saved metadata; shards not checked               10       
                    ESM-2 35M                 saved metadata; shards not checked               10       
                    ESM-2 150M                saved metadata; shards not checked               10       
                    ProtBERT                  saved metadata; shards not checked               10       
fungal_expression   NT v2 100M multi-species  saved metadata; shards not checked               13       
                    RNA-FM                    saved metadata; shards not checked               13       
                    ESM-2 8M                  saved metadata; shards not checked               13       
                    NT 500M human-ref         saved metadata; shards not checked               13       
                    HyenaDNA large            saved metadata; shards not checked               13       
                    RiNALMo 150M              saved metadata; shards not checked               13       
                    mRNA-FM                   saved metadata; shards not checked               13       
                    CaLM                      saved metadata; shards not checked               13       
                    ESM-2 35M                 saved metadata; shards not checked               13       
                    ESM-2 150M                saved metadata; shards not checked               13       
                    ProtBERT                  saved metadata; shards not checked               13       
ecoli_proteins      NT v2 100M multi-species  saved metadata; shards not checked               18       
                    RNA-FM                    saved metadata; shards not checked               18       
                    ESM-2 8M                  saved metadata; shards not checked               18       
                    NT 500M human-ref         saved metadata; shards not checked               18       
                    HyenaDNA large            saved metadata; shards not checked               18       
                    RiNALMo 150M              saved metadata; shards not checked               18       
                    mRNA-FM                   saved metadata; shards not checked               18       
                    CaLM                      saved metadata; shards not checked               18       
                    ESM-2 35M                 saved metadata; shards not checked               18       
                    ESM-2 150M                saved metadata; shards not checked               18       
                    ProtBERT                  saved metadata; shards not checked               18       
mrna_stability      NT v2 100M multi-species  saved metadata; shards not checked               94       
                    RNA-FM                    saved metadata; shards not checked               94       
                    ESM-2 8M                  saved metadata; shards not checked               94       
                    NT 500M human-ref         saved metadata; shards not checked    

Saved scores remain readable without token caches. No recovery is attempted.


In [ ]:
if not READ_SAVED_RESULTS:
    # Historical shard evidence is optional; ordered-input identity is still required.
    recovery_path = CACHE_ROOT / "recovery-records.json"
    RECOVERY_RECORDS = (
        {
            recovery_record["key"]: recovery_record
            for recovery_record in json.loads(recovery_path.read_text())["caches"]
        }
        if recovery_path.exists() else {}
    )

    # Share successful file checks with extraction until the file statistics change.
    CACHE_VERIFICATION = tokens.CacheVerification()
    CACHE_PLANS = {}
    BENCHMARK_ROWS = {}
    checks = []
    for benchmark in BENCHMARKS:
        needed_encoders = [
            encoder_key for encoder_key in required_encoders(benchmark)
            if (benchmark, encoder_key) in PENDING_CACHE_KEYS
        ]
        if not needed_encoders:
            continue
        rows = datasets.blf_benchmark_rows(benchmark)
        BENCHMARK_ROWS[benchmark] = rows
        for encoder_key in needed_encoders:
            encoder = ENCODERS[encoder_key]
            texts = tokens.inputs_for_rows(encoder, rows["sequence"].tolist())
            plan = tokens.inspect_cache(
                encoder,
                texts,
                CACHE_ROOT / benchmark / encoder_key,
                chunk_size=CHUNK_SIZE,
                dtype=DTYPE,
                recovery_evidence=RECOVERY_RECORDS.get(f"{benchmark}/{encoder_key}"),
                verification=CACHE_VERIFICATION,
            )
            CACHE_PLANS[benchmark, encoder_key] = plan
            checks.append({
                "benchmark": benchmark,
                "encoder": encoder.label,
                "status": plan.status,
                "reusable shards": len(plan.reusable_shards),
                "missing shards": len(plan.missing_shards),
                "input provenance": plan.provenance,
                "issues": "; ".join(plan.issues),
            })
            print(f"  {benchmark}/{encoder_key}: {plan.status}, {plan.provenance}", flush=True)
    if checks:
        display(pd.DataFrame(checks).set_index(["benchmark", "encoder"]))
    else:
        print("All requested fits are complete; cache verification is unnecessary.")

    # Stop before loading models if any cache cannot safely be reused or extended.
    blocked = [plan.summary for plan in CACHE_PLANS.values() if not plan.ready]
    if blocked:
        raise tokens.CacheRecoveryRequired(
            "Cache preflight stopped before model loading:\n" + "\n".join(blocked)
        )
    missing = sum(len(plan.missing_shards) for plan in CACHE_PLANS.values())
    print(f"Preflight passed: {len(CACHE_PLANS)} caches; {missing} shards need extraction.")


**Reading the computation-mode preflight table.** Each row describes one benchmark and encoder cache. A shard is a saved chunk of token embeddings; shard counts describe stored inputs rather than trained models.

| Field | Meaning |
| --- | --- |
| `status` | `complete` means all expected shards are present and the preflight reports no issues. It describes embedding readiness, not completion of downstream training. |
| `reusable shards` | Existing chunks that passed the applicable reuse checks. |
| `missing shards` | Expected chunks that are absent and would need extraction. |
| `input provenance` | `recorded_input` means the cache stores an identity for its ordered input sequences. Successful preflight with no issues establishes that this identity matches the requested inputs. |
| `issues` | Problems that block reuse and need inspection before extraction or training. |

When every required cache is complete, has no missing shards and reports no issues, no new embedding computation is needed. The reuse cell below opens only the caches needed by pending fits and reports what it found.

### Extract missing shards and reuse complete caches

For pending fits, open complete caches and compute only missing chunks permitted by the preflight. The original streams are included, so a missing original cache cannot be overlooked before training. Complete caches can be reused without loading an encoder; missing work may require model downloads and embedding computation.

Extraction records input identity before writing its first shard and checks token counts and finite values before publishing each new shard. Finished shards survive a failure; an unfinished shard may need to be repeated. A printed sequence count is progress, while a published shard is recoverable work. Cache and best-weight files remain local and ignored by Git.

The resulting table distinguishes extraction from reuse. Its GB column reports stored token payload, and the elapsed time describes that particular invocation and its mix of reused and computed work.

In [14]:
written = []
if READ_SAVED_RESULTS:
    print("Extraction skipped. Missing caches or shards are not computed in read-saved mode.")
else:
    started = time.perf_counter()
    for benchmark in BENCHMARKS:
        needed_encoders = [
            encoder_key for encoder_key in required_encoders(benchmark)
            if (benchmark, encoder_key) in PENDING_CACHE_KEYS
        ]
        if not needed_encoders:
            continue
        rows = BENCHMARK_ROWS[benchmark]
        for encoder_key in needed_encoders:
            encoder = ENCODERS[encoder_key]
            texts = tokens.inputs_for_rows(encoder, rows["sequence"].tolist())
            cache, computed = tokens.extract(
                encoder, texts, CACHE_ROOT / benchmark / encoder_key, DEVICE,
                chunk_size=CHUNK_SIZE, dtype=DTYPE,
                recovery_evidence=RECOVERY_RECORDS.get(f"{benchmark}/{encoder_key}"),
                verification=CACHE_VERIFICATION,
            )
            try:
                written.append({
                    "benchmark": benchmark, "encoder": encoder.label, "sequences": len(cache),
                    "tokens": cache.meta["total_tokens"], "width": cache.width,
                    "GB": round(cache.meta["bytes"] / 1e9, 2),
                    "this run": "extracted" if computed else "already complete",
                })
            finally:
                cache.close()
    if written:
        display(pd.DataFrame(written).set_index(["benchmark", "encoder"]))
    print(f"Extraction/reuse checks finished in {(time.perf_counter() - started) / 60:.1f} minutes.")


Extraction skipped. Missing caches or shards are not computed in read-saved mode.


**Reading the embedding inventory.** The table above describes the cached numerical features available to the prediction models.

| Field | Meaning |
| --- | --- |
| `sequences` | Benchmark rows represented in the cache, including repeated sequences if the benchmark contains them. |
| `tokens` | Total represented positions across those rows. Tokenization differs by encoder, so encoders can represent the same sequences with different token counts. |
| `width` | The number of values in each token's embedding vector. |
| `GB` | Stored token-array payload in decimal gigabytes, calculated from token count, width and dtype size. It excludes archive and metadata overhead and is not a peak RAM or GPU-memory requirement. |
| `this run` | `already complete` means this invocation reused the cache without loading the encoder. `extracted` means it computed missing chunks, which may be only part of the cache. |

Complete embeddings supply the inputs for fitting; they do not show whether prediction models have been trained or how accurately those models predict the benchmark outcome. In Part 2, a pending training group opens its token streams and checks cached row counts and encoder-specific token lengths. The earlier preflight supplies ordered-input identity checks; matching token lengths alone does not establish sequence identity. Completed fits open no token streams.


In [15]:
# Each training group owns and closes its token streams; completed fits open none.
print("Token streams open only for a pending training group.")


Token streams open only for a pending training group or explicit interval analysis.


---

## Part 2: Fit the encoder comparisons

### Prepare the training groups

**Why training follows extraction.** The pretrained encoders have converted sequences into numerical features. Training here keeps those embeddings fixed and learns the prediction head, together with any fusion and grid parameters used by that variant, to predict the benchmark's measured outcome. The pretrained encoders themselves are not fine-tuned.

The comparisons are the scientific purpose: does replacing one encoder improve the overall predictor, and does combining modalities add predictive value beyond the strongest individual encoder in that triple? For example, a replacement RNA encoder could improve both its single-encoder baseline and the fused model. Comparing fusion with the best single tests whether fusion adds predictive value beyond the strongest individual stream; comparing the fused model with the original triple measures the overall effect of the swap.

Keep the encoder triple fixed when comparing fusion heads: that comparison addresses the combination method, whereas comparing different triples also changes the representations. A head difference remains conditional on its capacity and inherited tuning; it does not identify a biological mechanism.

For each benchmark and replacement, fit the swapped-in encoder alone. When its stream is pooled or upsampled onto the grid, also fit the same encoder on that grid. Then compare concatenation, the gated MIL head and cross-attention using the replacement in its modality's slot.

| Variant in the code | What learns to predict the target | Why include it? |
| --- | --- | --- |
| `unimodal:<slot>` | TextCNN reads one encoder at its native token resolution | Measures what the replacement supports on its own |
| `unimodal-grid:<slot>` | The selected stream is placed on the codon grid before TextCNN | Checks whether changing resolution affects the single-encoder reference |
| `concat` | Features from all three streams are joined at each grid position and projected before TextCNN | Tests combining the representations without a learned modality gate |
| `mil` | A learned gate assigns one weight per modality per sequence, then combines the streams before TextCNN | Tests adaptive weighting across modalities; these weights are not codon-specific |
| `cross` | Each stream attends to the joint three-stream context before the updated streams are merged and read by TextCNN | Tests interactions between positions and modalities |

The gated model's **entropy weight**, `entropy_lambda`, controls a training penalty on how spread out its modality weights are. The implemented loss adds positive entropy, so minimizing that term favors concentrated weights. Phase B reuses the value selected on validation for the original encoders. It does not retune the replacement or establish that a large gate weight means biological importance. See the [fusion implementations](../shared_code/fusion/heads.py) and [training loss](../shared_code/fusion/train.py).

In [ ]:
def open_fit_streams(benchmark, encoder_mapping):
    """Open one triple, releasing earlier caches if a later open/check fails."""
    rows = BENCHMARK_ROWS.get(benchmark)
    if rows is None:
        rows = datasets.blf_benchmark_rows(benchmark)
    spec = datasets.DATASETS[benchmark]
    caches = {}
    try:
        for modality, encoder_key in encoder_mapping.items():
            caches[modality] = tokens.summarize(CACHE_ROOT / benchmark / encoder_key)
        streams = blf.BenchmarkStreams(
            benchmark, rows, caches, spec.task, spec.label_column, spec.split_column,
            encoders=dict(encoder_mapping),
        )
        streams.check()
        return streams
    except BaseException as error:
        for cache in caches.values():
            try:
                cache.close()
            except BaseException as cleanup_error:
                if hasattr(error, "add_note"):
                    error.add_note(f"Cache cleanup also failed: {cleanup_error}")
        raise


In [16]:
def run_pending_group(benchmark, encoder_key, planned_runs):
    """Run only new/resumable fits for one triple; reuse needs no token arrays."""
    pending = [run for run in planned_runs if run.status in ("new", "resume")]
    if not pending:
        return
    streams = open_fit_streams(benchmark, pending[0].encoders)
    primary_error = None
    try:
        for run in pending:
            print(f"{benchmark} / {encoder_key or 'original references'} / {run.variant} / seed {SEED}")
            train.run(
                streams, run.variant, train.TrainConfig(**run.config), DEVICE,
                run.path.parent, tag=run.path.stem,
                run_identity=requested_identity(
                    benchmark, run.variant, run.config, run.encoders, rows=streams.rows,
                    verification=CACHE_VERIFICATION,
                ),
                producer=run_identity.producer_record(device=DEVICE, repository=CODE_DIR.parent),
            )
    except BaseException as error:
        primary_error = error
        raise
    finally:
        try:
            streams.close()
        except BaseException as cleanup_error:
            if primary_error is None:
                raise
            if hasattr(primary_error, "add_note"):
                primary_error.add_note(f"Stream cleanup also failed: {cleanup_error}")

### Run the baselines and fusion heads

**Execution order.** Missing same-seed original references run first: native DNA/RNA/protein predictors, the original RNA grid control and the compared fusion heads. These are listed explicitly in the plan because a replacement must be compared with references fitted at the same seed. Compatible existing references are reused at their actual paths, including historical native-baseline filenames. The replacement sweep then retains its first pass of native/grid singles, concatenation and MIL, followed by cross-attention.

**Reuse and recovery.** The metadata plan checks the full training recipe, fixed selection prerequisites, duplicate identities, destination conflicts and recoverable checkpoints before extraction. It refreshes after extraction. The strict token preflight checks ordered inputs and cache compatibility only for pending fits. A compatible completed result is never fitted again to obtain missing predictions or weights. An incomplete or conflicting publication is preserved and reported instead of overwritten. These checks do not reconstruct unknown historical code or input identity.

**Resources and interruptions.** One triple is opened for a pending training group and closed on completion or failure; the notebook no longer keeps every benchmark/replacement's mappings open. Existing epoch checkpoints retain optimizer, scheduler, random-generator and batch-order state. Compatible checkpoints resume from their saved boundary, so unfinished epochs after that boundary may repeat; completed result records do not. Checkpoints missing required recovery evidence stop in preflight. Numerical trajectories can still vary across hardware.

The elapsed time applies to this invocation. No overnight duration or peak-memory guarantee is implied for the expanded stability workload.


In [17]:
if READ_SAVED_RESULTS:
    print("Training skipped. The execution plan reports reusable, missing and blocked fits.")
else:
    started = time.perf_counter()
    # Recheck destinations after extraction, so a completed canonical or renamed
    # result discovered since preflight is also reused rather than fitted again.
    FIT_PLANS = {benchmark: execution_plan(benchmark) for benchmark in BENCHMARKS}
    for plan in FIT_PLANS.values():
        plan.require_ready()

    # New seeds need same-seed original references as well as replacement fits.
    # Their missing entries are explicit in the plan, never inferred as completed.
    for benchmark, plan in FIT_PLANS.items():
        original_runs_to_fit = [run for run in plan.runs if run.role == "original"]
        run_pending_group(benchmark, None, original_runs_to_fit)

    # Preserve the replacement sweep's cheap-first, cross-attention-second order.
    for run_cross_attention in (False, True):
        for benchmark, plan in FIT_PLANS.items():
            for modality, encoder_key in swaps_for(benchmark):
                replacement_runs = [
                    run for run in plan.runs
                    if run.role == "replacement"
                    and run.encoder_key == encoder_key
                    and (run.variant == "cross") == run_cross_attention
                ]
                run_pending_group(benchmark, encoder_key, replacement_runs)
    print(f"Requested seed finished in {(time.perf_counter() - started) / 3600:.2f} hours.")


Training skipped. The execution plan reports reusable, missing and blocked fits.


---

## Part 3: Interpret the comparisons

For each regression benchmark, training minimizes mean squared error on the stored target scale, while the reported Spearman correlation measures predicted ordering. The encoder comparison therefore asks whether a replacement improves ranking of that benchmark's labels. Higher mRNA-stability labels indicate relatively greater stability; higher CoV-Vac labels indicate greater degradation. Neither score is a measurement of hours of persistence or vaccine performance, and fungal expression concerns RNA abundance rather than directly measured protein production.

For E. coli, accuracy measures how often the predicted low, medium or high expression category agrees with the recorded category. A score gain is an improvement in category prediction, not a proportional increase in protein yield. Compare each replacement with the original encoder on the same benchmark and test rows; differences between benchmarks also reflect their different targets, assays and evaluation tasks.


### Compare each swap with its baselines

Each benchmark table starts with BioLangFusion's original triple, followed by the replacements. The single-encoder scores belong to the three encoders in that row's triple. Seed-0 validation selects the “best single,” including a gridded reading where one is available. Seeds 1 and 2 reuse that selected variant and display its own same-seed test score.

**Two different questions.** Δfusion is a fusion head's test score minus its triple's validation-selected best single. “vs original” compares that fusion head with the same head using the original encoders on the same test rows. A replacement can improve the single encoder without increasing the benefit from fusion; read both differences together.

For example, an illustrative fused score of 0.85 against its own single-encoder reference of 0.84 gives a fusion gain of 0.01. If the original fused model scored 0.79, the same result gives a replacement gain of 0.06. The larger number describes the improvement from changing the encoder; it is not the additional contribution of fusion. These illustrative values are not another experimental result.

The `alone` column is the swapped encoder's native-resolution test score. `best single` is selected using validation, so it can name another stream or a grid baseline, and need not have the highest observed test score. Selecting a new winner after reading the test table would change the evaluation rule. E. coli scores use percent accuracy and differences use percentage points; regression differences use Spearman-correlation units.

The reader selects the configured training seed and full training recipe before constructing each triple. Recorded encoder mappings are checked when present; legacy files without them retain that provenance limit. Duplicate identities are withheld from reading and block computation; unrelated valid scores remain readable. Original native-resolution baselines bypass the codon-grid transposed convolution while retaining TextCNN: a matching convolution label is preferred, with a unique older label accepted when no matching record exists. Grid and fusion fits must match the requested convolution. If required single-encoder references are missing, the best-single score and its fusion gain remain unavailable.

In [18]:
# E. coli accuracy is displayed as a percentage; score differences are percentage points.
SCALE = {"ecoli_proteins": 100.0}
HEADS = swap_results.HEADS


def swap_runs(benchmark, encoder_key, mil_penalty, seed=None):
    """Read only the selected seed and inherited recipe, refusing duplicate identities."""
    modality = next(
        slot for slot, replacement_encoder in swaps_for(benchmark)
        if replacement_encoder == encoder_key
    )
    report = swap_report(
        benchmark, modality, encoder_key, SEED if seed is None else seed, mil_penalty,
    )
    records = report.records if READ_SAVED_RESULTS else report.require_unique()
    return [train.Result.load(result["path"]) for result in records]


def results_for_encoder_set(benchmark, modality=None, encoder_key=None, seed=None):
    """Collect unique variant records for the original or a replacement triple."""
    try:
        mil_penalty = selected_mil_penalty(benchmark)
    except RuntimeError:
        if not READ_SAVED_RESULTS:
            raise
        # Missing MIL selection does not hide independent single/concat/cross scores.
        mil_penalty = None
    runs = {}
    # Keep the validation-selected positive MIL penalty among the original runs.
    for result in original_runs(benchmark, seed):
        name = result_variant_label(result)
        if name == "mil" and result.config["entropy_lambda"] != mil_penalty:
            continue
        runs[name] = result
    if encoder_key is None:
        return runs

    # Retain unchanged-slot singles; replace this slot's baselines and compared heads.
    for name in [
        variant for variant in runs
        if variant in HEADS or variant.endswith(f":{modality}")
    ]:
        del runs[name]
    for result in swap_runs(benchmark, encoder_key, mil_penalty, seed):
        runs[result_variant_label(result)] = result
    return runs


In [ ]:
def validation_selected_single(runs, modality=None, encoder_key=None, selection_runs=None):
    """Select on validation only when the required single-encoder references exist."""
    required = {f"unimodal:{slot}" for slot in ORIGINAL}
    if encoder_key and tokens.grid_resolution(encoder_key) != "codon":
        required.add(f"unimodal-grid:{modality}")
    selection_runs = runs if selection_runs is None else selection_runs
    if not required.issubset(selection_runs):
        return None
    # Available historical grid controls remain eligible; missing native references
    # must not make an incomplete set look like a fully compared "best single."
    singles = {
        variant: result for variant, result in selection_runs.items()
        if variant.startswith("unimodal")
    }
    selected_variant = max(singles, key=lambda variant: singles[variant].val_score)
    return selected_variant if selected_variant in runs else None

**Build the comparison rows.** `benchmark_score_table` puts the native single-encoder score, the validation-selected reference, each fusion score, and the two kinds of difference in one row per replacement. Missing results remain missing rather than becoming zero scores.

In [19]:
def benchmark_score_table(benchmark):
    """Build one benchmark's score and gain rows from saved results."""
    scale = SCALE.get(benchmark, 1.0)
    original = results_for_encoder_set(benchmark)
    rows = []
    # The (None, None) entry supplies the original-triple reference row first.
    for modality, encoder_key in [(None, None), *swaps_for(benchmark)]:
        runs = results_for_encoder_set(benchmark, modality, encoder_key)
        selection_runs = (
            runs if SEED == SELECTION_SEED
            else results_for_encoder_set(
                benchmark, modality, encoder_key, seed=SELECTION_SEED,
            )
        )
        selected_single = validation_selected_single(
            runs, modality, encoder_key, selection_runs,
        )
        # No replacement-only score exists for the original row; unavailable scores stay NaN.
        row = {
            "slot": modality or "none",
            "encoder": (
                ENCODERS[encoder_key].label if encoder_key else "BioLangFusion's own"
            ),
            "alone": (
                runs.get(f"unimodal:{modality}").test_score * scale
                if modality and f"unimodal:{modality}" in runs else np.nan
            ),
            "best single": selected_single,
            "best single score": (
                runs[selected_single].test_score * scale if selected_single else np.nan
            ),
        }
        for head in HEADS:
            score = runs[head].test_score * scale if head in runs else np.nan
            row[head] = score
            # Distinguish fusion benefit from the overall effect of replacing an encoder.
            row[f"{head} Δfusion"] = score - row["best single score"]
            row[f"{head} vs original"] = (
                score - original[head].test_score * scale
                if head in original and encoder_key else np.nan
            )
        rows.append(row)
    return pd.DataFrame(rows).set_index(["slot", "encoder"])

**Read the benchmark tables.** The following cell reads saved results and displays each benchmark separately. Compare scores within a benchmark, then use the paired intervals below to assess the differences.

In [20]:
for benchmark in BENCHMARKS:
    print(f"\n{benchmark}" + (" (accuracy x 100)" if benchmark in SCALE else " (Spearman)"))
    try:
        display(benchmark_score_table(benchmark).round(3))
    except (OSError, RuntimeError, ValueError, KeyError, TypeError) as missing:
        print("  not ready:", missing)


cov_vac_degradation (Spearman)


alone   best single  best single score  concat  concat Δfusion  concat vs original    mil  mil Δfusion  mil vs original  cross  cross Δfusion  cross vs original
slot encoder                                                                                                                                                                              
none BioLangFusion's own    NaN  unimodal:rna              0.833   0.801          -0.032                 NaN  0.783       -0.050              NaN  0.832         -0.001                NaN
dna  NT 500M human-ref    0.727  unimodal:rna              0.833   0.753          -0.080              -0.048  0.710       -0.123           -0.073  0.811         -0.022             -0.021
     HyenaDNA large       0.755  unimodal:rna              0.833   0.832          -0.001               0.030  0.797       -0.036            0.014  0.872          0.039              0.040
rna  RiNALMo 150M         0.823  unimodal:rna              0.823   0.806          -0.017               0.004  0.819       -0.004            0.036  0.854          0.031              0.021
     mRNA-FM              0.812  unimodal:rna              0.812   0.759          -0.054              -0.043  0.762       -0.050           -0.021  0.822          0.009             -0.011
     CaLM                 0.844  unimodal:rna              0.844   0.817          -0.027               0.016  0.802       -0.042            0.019  0.828         -0.017             -0.005
prot ESM-2 35M            0.814  unimodal:rna              0.833   0.808          -0.025               0.007  0.793       -0.040            0.010  0.851          0.018              0.018
     ESM-2 150M           0.827  unimodal:rna              0.833   0.802          -0.031               0.001  0.783       -0.050            0.000  0.853          0.020              0.020
     ProtBERT             0.776  unimodal:rna              0.833   0.788          -0.045              -0.013  0.791       -0.043            0.007  0.843          0.010              0.011


fungal_expression (Spearman)


alone        best single  best single score  concat  concat Δfusion  concat vs original    mil  mil Δfusion  mil vs original  cross  cross Δfusion  cross vs original
slot encoder                                                                                                                                                                                   
none BioLangFusion's own    NaN       unimodal:dna              0.813   0.793          -0.020                 NaN  0.802       -0.011              NaN  0.824          0.011                NaN
dna  NT 500M human-ref    0.740  unimodal-grid:rna              0.794   0.794           0.000               0.002  0.791       -0.003           -0.011  0.810          0.016             -0.014
     HyenaDNA large       0.737  unimodal-grid:rna              0.794   0.786          -0.007              -0.006  0.780       -0.014           -0.022  0.804          0.010             -0.020
rna  RiNALMo 150M         0.844       unimodal:rna              0.844   0.846           0.002               0.054  0.831       -0.013            0.029  0.847          0.003              0.023
     mRNA-FM              0.852       unimodal:rna              0.852   0.853           0.001               0.061  0.839       -0.013            0.037  0.840         -0.012              0.017
     CaLM                 0.851       unimodal:rna              0.851   0.848          -0.003               0.056  0.848       -0.003            0.046  0.857          0.006              0.034
prot ESM-2 35M            0.696       unimodal:dna              0.813   0.795          -0.018               0.002  0.780       -0.033           -0.022  0.819          0.006             -0.005
     ESM-2 150M           0.729       unimodal:dna              0.813   0.784          -0.028              -0.008  0.784       -0.028           -0.018  0.811         -0.002             -0.013
     ProtBERT             0.761      unimodal:prot              0.761   0.805           0.044               0.012  0.800        0.039           -0.002  0.816          0.054             -0.008


ecoli_proteins (accuracy x 100)


alone        best single  best single score  concat  concat Δfusion  concat vs original     mil  mil Δfusion  mil vs original   cross  cross Δfusion  cross vs original
slot encoder                                                                                                                                                                                      
none BioLangFusion's own     NaN      unimodal:prot             50.140  60.674          10.534                 NaN  54.354        4.213              NaN  53.792          3.652                NaN
dna  NT 500M human-ref    47.472      unimodal:prot             50.140  57.865           7.725              -2.809  54.213        4.073           -0.140  55.337          5.197              1.545
     HyenaDNA large       50.421      unimodal:prot             50.140  58.006           7.865              -2.669  53.090        2.949           -1.264  51.966          1.826             -1.826
rna  RiNALMo 150M         57.303  unimodal-grid:rna             56.039  59.972           3.933              -0.702  62.500        6.461            8.146  54.354         -1.685              0.562
     mRNA-FM              58.708       unimodal:rna             58.708  58.989           0.281              -1.685  53.090       -5.618           -1.264  54.775         -3.933              0.983
     CaLM                 57.584       unimodal:rna             57.584  57.725           0.140              -2.949  56.742       -0.843            2.388  53.933         -3.652              0.140
prot ESM-2 35M            52.107      unimodal:prot             52.107  57.725           5.618              -2.949  53.230        1.124           -1.124  53.230          1.124             -0.562
     ESM-2 150M           53.933      unimodal:prot             53.933  57.725           3.792              -2.949  55.478        1.545            1.124  54.635          0.702              0.843
     ProtBERT             52.528      unimodal:prot             52.528  54.213           1.685              -6.461  56.461        3.933            2.107  54.635          2.107              0.843


mrna_stability (Spearman)


alone        best single  best single score  concat  concat Δfusion  concat vs original    mil  mil Δfusion  mil vs original  cross  cross Δfusion  cross vs original
slot encoder                                                                                                                                                                                   
none BioLangFusion's own    NaN       unimodal:rna              0.552   0.558           0.007                 NaN  0.551       -0.001              NaN  0.553          0.001                NaN
dna  NT 500M human-ref    0.537       unimodal:rna              0.552   0.552          -0.000              -0.007  0.553        0.002            0.003  0.545         -0.006             -0.007
     HyenaDNA large       0.523       unimodal:rna              0.552   0.557           0.005              -0.001  0.539       -0.012           -0.011  0.548         -0.004             -0.005
rna  RiNALMo 150M         0.560  unimodal-grid:rna              0.556   0.556           0.000              -0.002  0.564        0.008            0.014  0.558          0.003              0.006
     mRNA-FM              0.565       unimodal:rna              0.565   0.568           0.003               0.010  0.565       -0.000            0.014  0.562         -0.003              0.009
     CaLM                 0.559       unimodal:rna              0.559   0.561           0.002               0.003  0.561        0.002            0.010  0.548         -0.010             -0.004
prot ESM-2 35M            0.535       unimodal:rna              0.552   0.554           0.003              -0.004  0.551       -0.000            0.001  0.541         -0.011             -0.012
     ESM-2 150M           0.536       unimodal:rna              0.552   0.559           0.008               0.001  0.549       -0.003           -0.002  0.559          0.007              0.006
     ProtBERT             0.543      unimodal:prot              0.543   0.558           0.015              -0.001  0.550        0.008           -0.000  0.551          0.009             -0.001

**What the saved score tables suggest.** The fungal RNA replacements have strong single-encoder scores, while their extra gains from fusion are small or negative in several rows. CoV-Vac and E. coli also contain positive fusion differences for particular replacement/head combinations. These patterns answer different questions from a universal encoder ranking: both the biological target and the downstream model matter.

Point differences alone do not tell us which directions the evaluated test rows support. The next section compares each fitted model with its declared reference using those same rows, before returning to the worked RNA-FM/mRNA-FM example.

### Assess what the paired differences resolve

The paired bootstrap compares predictions on the same test rows. For each head and replacement, it estimates the difference from the original-encoder head and the difference from the replacement triple's best single. Exact duplicate sequences are grouped during resampling.

**Interpretation.** Read each interval against zero. An interval spanning zero leaves the direction unresolved under this procedure; it does not establish equivalence. Test-set sizes differ across the four benchmarks, so neither a universal row-count range nor a fixed 0.02 score threshold is justified. These intervals describe uncertainty from the evaluated rows, not variation across training seeds or protection against train/test sequence overlap.

The first helper, `bootstrap_comparison_plan`, names the saved models and eligible comparisons. The following helper, `paired_bootstrap_differences`, validates their predictions and requests 2,000 resamples with bootstrap seed 0. The bootstrap seed controls resampling independently of the training seed, `SEED`. Despite the helper name `seed_averaged_bootstrap`, each model is supplied as a list containing one saved fit in this historical route.

| Output | How to read it |
| --- | --- |
| `difference` | First named model's score minus its reference on the evaluated test rows |
| `ci_low`, `ci_high` | 2.5th and 97.5th percentiles of the paired resampled differences |
| `first_wins` | Fraction of resamples with a positive difference; not a p-value or the probability that a biological hypothesis is true |
| `resolved` | Whether the interval excludes zero in either direction; `yes` can therefore mean a loss as well as a gain |

These are individual comparison intervals without a correction for examining many swaps and heads. They describe uncertainty conditional on the saved fitted models and evaluation procedure.

Before resampling, the reader requires saved predictions and checks their row counts and saved target order against the current test targets at the loader's label precision (float32 for regression, int64 for classification). The bootstrap itself keeps its existing target array. This detects missing or mismatched arrays; equal target values alone do not prove sequence identity where row-level provenance was never recorded. Score-only historical results remain usable in the preceding score table.

In [ ]:
def bootstrap_comparison_plan(benchmark):
    """Name the saved records and eligible original/best-single comparisons."""
    original = results_for_encoder_set(benchmark)
    records, comparisons = {}, []

    for name, result in original.items():
        records[f"original {name}"] = result
    for modality, encoder_key in swaps_for(benchmark):
        runs = results_for_encoder_set(benchmark, modality, encoder_key)
        selection_runs = (
            runs if SEED == SELECTION_SEED
            else results_for_encoder_set(
                benchmark, modality, encoder_key, seed=SELECTION_SEED,
            )
        )
        selected_single = validation_selected_single(
            runs, modality, encoder_key, selection_runs,
        )
        for name, result in runs.items():
            records[f"{encoder_key} {name}"] = result
        for head in HEADS:
            if head not in runs or head not in original:
                continue
            # Each tuple names the displayed comparison, first predictor, and reference.
            comparisons.append((
                f"{encoder_key} {head} vs original",
                f"{encoder_key} {head}",
                f"original {head}",
            ))
            if selected_single is not None:
                comparisons.append((
                    f"{encoder_key} {head} vs best single ({selected_single})",
                    f"{encoder_key} {head}",
                    f"{encoder_key} {selected_single}",
                ))
    return records, comparisons


In [21]:
def paired_bootstrap_differences(benchmark, draws=2000):
    """Compare saved predictions on paired, sequence-grouped test resamples."""
    # Prediction pairing needs the benchmark's labels/order, not any token arrays.
    rows = BENCHMARK_ROWS.get(benchmark)
    if rows is None:
        rows = datasets.blf_benchmark_rows(benchmark)
    spec = datasets.DATASETS[benchmark]
    test = np.flatnonzero(rows[spec.split_column].to_numpy() == "test")
    truth = rows[spec.label_column].to_numpy()[test]
    task = spec.task
    target_dtype = np.int64 if task == "classification" else np.float32
    saved_target_order = np.asarray(truth, dtype=target_dtype).tolist()
    groups = rows["sequence"].to_numpy()[test]
    n_outputs = int(rows[spec.label_column].max()) + 1 if task == "classification" else None
    records, comparisons = bootstrap_comparison_plan(benchmark)
    predictions = {}

    # Validate only records used by a comparison: score-only historical baselines
    # remain readable, but missing/misordered predictions cannot support a paired interval.
    needed = {name for _, first, second in comparisons for name in (first, second)}
    for name in sorted(needed):
        result = records[name]
        issue = swap_results.prediction_issue(
            vars(result), saved_target_order, task=task,
            n_classes=n_outputs,
        )
        if issue:
            raise ValueError(f"{benchmark}, {name}: {issue}; paired comparison not calculated")
        predictions[name] = [np.asarray(result.test_prediction)]
    if not comparisons:
        raise ValueError(f"{benchmark}: no compatible paired comparisons; inspect result coverage")

    # Each prediction list contains one fitted run; this is not a multi-seed average here.
    out = pd.DataFrame(
        train.seed_averaged_bootstrap(
            task,
            truth,
            predictions,
            comparisons,
            groups=groups,
            draws=draws,
            seed=0,
        )
    )

    # Convert score differences to percentage points for classification, not first_wins.
    scale = SCALE.get(benchmark, 1.0)
    for column in ("difference", "ci_low", "ci_high"):
        out[column] = out[column] * scale
    # Excluding zero resolves either a gain or a loss; retain the difference's sign.
    out["resolved"] = np.where(
        (out["ci_low"] > 0) | (out["ci_high"] < 0),
        "yes",
        "no",
    )
    return out.set_index("comparison")[[
        "difference", "ci_low", "ci_high", "first_wins", "resolved",
    ]]

**Historical interval route.** This single-training-seed calculation runs only when `READ_SAVED_RESULTS = False` and `RUN_BOOTSTRAP = True`. That is computation mode: earlier cells may fit pending models before reaching the intervals. Leave `RUN_BOOTSTRAP = False` when using the saved-comparison report that follows. That report writes a separate, reusable analysis artifact and includes method-versus-concatenation contrasts and explicit seed coverage. Retained intervals describe the execution that produced them.


In [22]:
RESOLVED = {}
if READ_SAVED_RESULTS or not RUN_BOOTSTRAP:
    print("Paired intervals not recalculated. RUN_BOOTSTRAP is separate from completing fits; "
          "retained interval outputs remain historical.")
else:
    for benchmark in BENCHMARKS:
        try:
            RESOLVED[benchmark] = paired_bootstrap_differences(benchmark)
        except (RuntimeError, ValueError, KeyError) as missing:
            print(f"{benchmark}: not ready ({missing})")
            continue
        print(f"\n{benchmark}")
        display(RESOLVED[benchmark].round(3))


Paired intervals not recalculated. RUN_BOOTSTRAP is separate from completing fits; retained interval outputs remain historical.


### Report representation, complementarity and method differences

This optional analysis reads completed result JSONs and the benchmark's test labels and sequence groups. It never opens token embeddings or fits a model. In Setup, keep `READ_SAVED_RESULTS = True` and set `RUN_SAVED_COMPARISONS = True`; leave `RUN_BOOTSTRAP = False`. Run from the top after restarting the kernel. Missing fits or unusable predictions leave the affected comparisons unavailable. Ambiguous result identities, an incomplete original seed-0 MIL tuning grid or incompatible report files stop reporting for inspection; none of these conditions triggers another fit.

The report separates the replacement single versus the original single, each replacement fusion head versus the corresponding original head, each fusion head versus its seed-0 validation-selected single, and gated/cross-attention versus concatenation of the same inputs. The original positive MIL penalty and single-reference choices remain fixed at seed 0. Neither selection uses test performance.

**Units and target precision.** E. coli model scores (`first_score`, `second_score`) use percent accuracy; differences, interval endpoints and seed spread use percentage points. Other benchmarks use Spearman-correlation units. Scores and resampled differences use the trainer's target precision (float32 for regression, integer class IDs for classification), including ties introduced when CSV values round to float32.

Each available seed receives its own paired comparison. The requested-seed summary requires every ID in `COMPARISON_SEEDS` for both sides: when a seed or prediction is missing, the mean and interval remain unavailable. When complete, the summary averages the per-seed scores and reports the sample standard deviation of paired seed differences. It does not average predictions into an ensemble. The group-bootstrap interval conditions on those fitted seeds and resamples exact test-sequence groups; it does not estimate training-seed uncertainty, remove train/test overlap or adjust for multiple comparisons. Undefined rank-correlation draws are counted and withhold the interval instead of silently discarding them.

New reports are stored in `results/phaseB/analyses/paired-report-v1/`. The JSON includes per-seed results, requested-seed coverage, intervals, source-file hashes, selection choices, current target/group identity and execution information. The content-identified destination is reused only for the same analysis inputs and settings; completed fit files and earlier reports are never overwritten. Matching current target values cannot recover missing historical sequence-row provenance, so that limitation remains in the report.


In [ ]:
def build_or_reuse_report(benchmark, reporting):
    """Calculate or reuse the immutable report from saved predictions and benchmark labels."""
    # This is label/sequence preparation only; no token cache or model is opened.
    rows = datasets.blf_benchmark_rows(benchmark)
    spec = datasets.DATASETS[benchmark]
    test_rows = np.flatnonzero(rows[spec.split_column].to_numpy() == "test")
    current_context = reporting.benchmark_context(
        rows, task=spec.task, label_column=spec.label_column,
        split_column=spec.split_column,
    )
    inventory = reporting.collect(
        benchmark=benchmark, config=result_recipe(0),
        original_folders=ORIGINAL_RESULTS[benchmark], original_encoders=ORIGINAL,
        swaps=[
            (modality, encoder_key, tokens.grid_resolution(encoder_key) != "codon")
            for modality, encoder_key in swaps_for(benchmark)
        ],
        swap_root=RESULTS_ROOT / benchmark, seeds=COMPARISON_SEEDS,
    )
    n_classes = (
        int(rows[spec.label_column].max()) + 1
        if spec.task == "classification" else None
    )
    saved, analysis_path, computed = reporting.report(
        inventory, truth=rows[spec.label_column].to_numpy()[test_rows],
        groups=rows["sequence"].to_numpy()[test_rows], task=spec.task,
        context=current_context, output_dir=COMPARISON_DIR,
        draws=COMPARISON_DRAWS, bootstrap_seed=COMPARISON_BOOTSTRAP_SEED,
        n_classes=n_classes,
        repository=CODE_DIR.parent,
    )
    return saved, analysis_path, computed


In [23]:
SAVED_COMPARISON_REPORTS = {}
if not RUN_SAVED_COMPARISONS:
    print("Saved-prediction report skipped. Enable RUN_SAVED_COMPARISONS in Setup "
          "with READ_SAVED_RESULTS = True to calculate or reuse it.")
else:
    if not READ_SAVED_RESULTS:
        raise ValueError("Saved-prediction reporting requires READ_SAVED_RESULTS = True.")
    from shared_code.fusion import swap_reporting

    for benchmark in BENCHMARKS:
        saved, analysis_path, computed = build_or_reuse_report(benchmark, swap_reporting)
        spec = datasets.DATASETS[benchmark]
        SAVED_COMPARISON_REPORTS[benchmark] = saved
        analysis = saved["analysis"]
        print(f"{benchmark}: {'calculated' if computed else 'reused'} {analysis_path}")
        print("Scores: percent accuracy; differences/intervals/spread: percentage points."
              if spec.task == "classification" else "Scores and differences: Spearman-correlation units.")
        per_seed = pd.DataFrame(analysis["per_seed"])
        if not per_seed.empty:
            display(per_seed.drop(columns=["first", "second"]).set_index(
                ["encoder", "kind", "comparison", "seed"]).round(4))
        display(pd.DataFrame(analysis["seed_summary"]).drop(columns=["first", "second"]).set_index(
            ["encoder", "kind", "comparison"]).round(4))
        if analysis["issues"]:
            display(pd.DataFrame(analysis["issues"]))


cov_vac_degradation: calculated D:\academic\UCF\2026_fall\Capstone Project CAP6942\Research\genomics-multimodal-fusion\Code\stage4\results\phaseB\analyses\paired-report-v1\cov_vac_degradation__9c0f130491382c1bc2ad.json
Scores and differences: Spearman-correlation units.


first_score  second_score  difference seed_sd  ci_low  ci_high  first_wins  valid_draws  requested_draws interval_issue
encoder  kind            comparison                            seed                                                                                                                         
nt500m   representation  replacement single vs original single 0          0.7269        0.7790     -0.0521    None -0.0980  -0.0051      0.0155         2000             2000               
         replacement     concat vs original concat             0          0.7531        0.8014     -0.0483    None -0.0805  -0.0141      0.0020         2000             2000               
         complementarity concat vs validation-selected single  0          0.7531        0.8332     -0.0801    None -0.1132  -0.0454      0.0000         2000             2000               
         replacement     mil vs original mil                   0          0.7103        0.7831     -0.0728    None -0.1163  -0.0312      0.0010         2000             2000               
         complementarity mil vs validation-selected single     0          0.7103        0.8332     -0.1229    None -0.1682  -0.0772      0.0000         2000             2000               
...                                                                          ...           ...         ...     ...     ...      ...         ...          ...              ...            ...
protbert complementarity mil vs validation-selected single     0          0.7906        0.8332     -0.0426    None -0.0785  -0.0076      0.0090         2000             2000               
         replacement     cross vs original cross               0          0.8434        0.8324      0.0109    None -0.0128   0.0366      0.8155         2000             2000               
         complementarity cross vs validation-selected single   0          0.8434        0.8332      0.0102    None -0.0219   0.0412      0.7370         2000             2000               
         method          mil vs concat                         0          0.7906        0.7883      0.0023    None -0.0250   0.0283      0.5625         2000             2000               
                         cross vs concat                       0          0.8434        0.7883      0.0550    None  0.0242   0.0879      1.0000         2000             2000               

[72 rows x 10 columns]

available_seeds missing_seeds      status first_score second_score difference seed_sd ci_low ci_high first_wins
encoder  kind            comparison                                                                                                                                           
nt500m   representation  replacement single vs original single             [0]        [1, 2]  incomplete        None         None       None    None   None    None       None
         replacement     concat vs original concat                         [0]        [1, 2]  incomplete        None         None       None    None   None    None       None
         complementarity concat vs validation-selected single              [0]        [1, 2]  incomplete        None         None       None    None   None    None       None
         replacement     mil vs original mil                               [0]        [1, 2]  incomplete        None         None       None    None   None    None       None
         complementarity mil vs validation-selected single                 [0]        [1, 2]  incomplete        None         None       None    None   None    None       None
...                                                                        ...           ...         ...         ...          ...        ...     ...    ...     ...        ...
protbert complementarity mil vs validation-selected single                 [0]        [1, 2]  incomplete        None         None       None    None   None    None       None
         replacement     cross vs original cross                           [0]        [1, 2]  incomplete        None         None       None    None   None    None       None
         complementarity cross vs validation-selected single               [0]        [1, 2]  incomplete        None         None       None    None   None    None       None
         method          mil vs concat                                     [0]        [1, 2]  incomplete        None         None       None    None   None    None       None
                         cross vs concat                                   [0]        [1, 2]  incomplete        None         None       None    None   None    None       None

[72 rows x 10 columns]

,model,seed,reason
0,calm/concat,1,completed fit absent
1,calm/concat,2,completed fit absent
2,calm/cross,1,completed fit absent
3,calm/cross,2,completed fit absent
4,calm/mil,1,completed fit absent
...,...,...,...
63,rinalmo/cross,2,completed fit absent
64,rinalmo/mil,1,completed fit absent
65,rinalmo/mil,2,completed fit absent
66,rinalmo/unimodal:rna,1,completed fit absent


fungal_expression: calculated D:\academic\UCF\2026_fall\Capstone Project CAP6942\Research\genomics-multimodal-fusion\Code\stage4\results\phaseB\analyses\paired-report-v1\fungal_expression__d2173df49ec55ed37414.json
Scores and differences: Spearman-correlation units.


first_score  second_score  difference seed_sd  ci_low  ci_high  first_wins  valid_draws  requested_draws interval_issue
encoder  kind            comparison                            seed                                                                                                                         
nt500m   representation  replacement single vs original single 0          0.7397        0.8125     -0.0728    None -0.1096  -0.0406      0.0000         2000             2000               
         replacement     concat vs original concat             0          0.7943        0.7926      0.0017    None -0.0243   0.0283      0.5180         2000             2000               
         complementarity concat vs validation-selected single  0          0.7943        0.7938      0.0005    None -0.0323   0.0338      0.5160         2000             2000               
         replacement     mil vs original mil                   0          0.7905        0.8018     -0.0113    None -0.0376   0.0150      0.1920         2000             2000               
         complementarity mil vs validation-selected single     0          0.7905        0.7938     -0.0033    None -0.0382   0.0312      0.4340         2000             2000               
...                                                                          ...           ...         ...     ...     ...      ...         ...          ...              ...            ...
protbert complementarity mil vs validation-selected single     0          0.8001        0.7612      0.0389    None  0.0034   0.0754      0.9850         2000             2000               
         replacement     cross vs original cross               0          0.8155        0.8235     -0.0080    None -0.0374   0.0194      0.3060         2000             2000               
         complementarity cross vs validation-selected single   0          0.8155        0.7612      0.0543    None  0.0328   0.0762      1.0000         2000             2000               
         method          mil vs concat                         0          0.8001        0.8049     -0.0048    None -0.0183   0.0085      0.2335         2000             2000               
                         cross vs concat                       0          0.8155        0.8049      0.0106    None -0.0131   0.0348      0.8090         2000             2000               

[66 rows x 10 columns]

... (6 outputs left out)

mrna_stability: calculated D:\academic\UCF\2026_fall\Capstone Project CAP6942\Research\genomics-multimodal-fusion\Code\stage4\results\phaseB\analyses\paired-report-v1\mrna_stability__d703113ac8fcdabfd4bd.json
Scores and differences: Spearman-correlation units.


first_score  second_score  difference seed_sd  ci_low  ci_high  first_wins  valid_draws  requested_draws interval_issue
encoder  kind            comparison                            seed                                                                                                                         
nt500m   representation  replacement single vs original single 0          0.5368        0.5391     -0.0023    None -0.0150   0.0099      0.3780         2000             2000               
         replacement     concat vs original concat             0          0.5515        0.5582     -0.0066    None -0.0171   0.0034      0.0985         2000             2000               
         complementarity concat vs validation-selected single  0          0.5515        0.5517     -0.0001    None -0.0129   0.0122      0.4960         2000             2000               
         replacement     mil vs original mil                   0          0.5533        0.5508      0.0025    None -0.0080   0.0126      0.6965         2000             2000               
         complementarity mil vs validation-selected single     0          0.5533        0.5517      0.0016    None -0.0118   0.0151      0.6005         2000             2000               
...                                                                          ...           ...         ...     ...     ...      ...         ...          ...              ...            ...
protbert complementarity mil vs validation-selected single     0          0.5504        0.5427      0.0077    None -0.0045   0.0202      0.9020         2000             2000               
         replacement     cross vs original cross               0          0.5514        0.5527     -0.0013    None -0.0147   0.0114      0.4115         2000             2000               
         complementarity cross vs validation-selected single   0          0.5514        0.5427      0.0086    None -0.0038   0.0214      0.9105         2000             2000               
         method          mil vs concat                         0          0.5504        0.5577     -0.0073    None -0.0178   0.0033      0.0925         2000             2000               
                         cross vs concat                       0          0.5514        0.5577     -0.0063    None -0.0183   0.0052      0.1235         2000             2000               

[72 rows x 10 columns]

available_seeds missing_seeds      status first_score second_score difference seed_sd ci_low ci_high first_wins
encoder  kind            comparison                                                                                                                                           
nt500m   representation  replacement single vs original single             [0]        [1, 2]  incomplete        None         None       None    None   None    None       None
         replacement     concat vs original concat                         [0]        [1, 2]  incomplete        None         None       None    None   None    None       None
         complementarity concat vs validation-selected single              [0]        [1, 2]  incomplete        None         None       None    None   None    None       None
         replacement     mil vs original mil                               [0]        [1, 2]  incomplete        None         None       None    None   None    None       None
         complementarity mil vs validation-selected single                 [0]        [1, 2]  incomplete        None         None       None    None   None    None       None
...                                                                        ...           ...         ...         ...          ...        ...     ...    ...     ...        ...
protbert complementarity mil vs validation-selected single                 [0]        [1, 2]  incomplete        None         None       None    None   None    None       None
         replacement     cross vs original cross                           [0]        [1, 2]  incomplete        None         None       None    None   None    None       None
         complementarity cross vs validation-selected single               [0]        [1, 2]  incomplete        None         None       None    None   None    None       None
         method          mil vs concat                                     [0]        [1, 2]  incomplete        None         None       None    None   None    None       None
                         cross vs concat                                   [0]        [1, 2]  incomplete        None         None       None    None   None    None       None

[72 rows x 10 columns]

,model,seed,reason
0,calm/concat,1,completed fit absent
1,calm/concat,2,completed fit absent
2,calm/cross,1,completed fit absent
3,calm/cross,2,completed fit absent
4,calm/mil,1,completed fit absent
...,...,...,...
81,rinalmo/mil,2,completed fit absent
82,rinalmo/unimodal-grid:rna,1,completed fit absent
83,rinalmo/unimodal-grid:rna,2,completed fit absent
84,rinalmo/unimodal:rna,1,completed fit absent


### What the saved comparisons establish

The following examples summarize the retained score and interval tables and the [October 27 readout](../../docs/run-records.md#2026-10-27-phaseb-complete-readout-all-eight-swaps-written-up-from-the-saved-notebook-outputs-no-new-fits). They are readings of existing seed-0 results, not new fits or recalculated intervals.

| Comparison | Saved observation | What it supports and what it leaves open |
| --- | --- | --- |
| CoV-Vac, HyenaDNA with cross-attention | Spearman 0.872; gain over the selected RNA-FM single of +0.039 [0.012, 0.066] | A positive fusion difference for this replacement/head on chemical-degradation labels; it does not establish a translation mechanism for these RNA fragments |
| Fungal expression, ProtBERT with cross-attention | Spearman 0.816; gain over the validation-selected ProtBERT single of +0.054 [0.033, 0.076] | A positive difference against the declared 0.761 reference; the retained NT v2 test score is 0.813, so this is not evidence of a large margin over every single encoder |
| E. coli, RiNALMo with gated MIL | Accuracy 62.500%; gain over the selected gridded RNA single of +6.461 [3.371, 9.551] percentage points | A positive fusion difference for expression-category prediction; the selected reference is the grid baseline, not RiNALMo's native-resolution score |
| mRNA stability, the three-replacement interval subset retained from the October 27 readout | Every interval in that subset includes zero | No resolved extra fusion benefit in this historical interval subset; later fit coverage does not extend these intervals to the other configured replacements |

The fungal ProtBERT example also distinguishes a larger fusion gain from a higher absolute score: the mRNA-FM concatenated model scores 0.853, above ProtBERT cross-attention's 0.816, but adds little over mRNA-FM alone. Keep the declared validation selection when interpreting these differences rather than replacing it with a test-selected reference.

These cases qualify a simple story that fusion never helps or that codon pretraining always wins. The evidence is specific to an encoder, head, target, reference, and seed. The worked RNA-FM/mRNA-FM comparison below examines those distinctions in more detail.

### 3. What the saved RNA-FM and mRNA-FM results show

The following reading uses the retained seed-0 outputs above and the [October 27 saved-result readout](../../docs/run-records.md#2026-10-27-phaseb-complete-readout-all-eight-swaps-written-up-from-the-saved-notebook-outputs-no-new-fits). It is not a new fit or recomputation. “Native” means the single-encoder predictor reads that encoder's token sequence before grid pooling; the RNA-FM grid control reads states averaged in threes.

| Benchmark and metric | RNA-FM native | RNA-FM grid control | mRNA-FM native | Concatenation gain over mRNA-FM alone [saved paired interval] |
| --- | ---: | ---: | ---: | --- |
| CoV-Vac degradation, Spearman | 0.833 | 0.823 | 0.812 | −0.054 [−0.088, −0.019] |
| Fungal expression, Spearman | 0.722 | 0.794 | 0.852 | +0.001 [−0.014, +0.016] |
| E. coli protein-expression class, accuracy | 50.70% | 49.30% | 58.71% | +0.28 percentage points [−2.95, +3.51] |
| mRNA stability, Spearman | 0.552 | 0.547 | 0.565 | +0.003 [−0.003, +0.010] |

These are within-benchmark comparisons, not one common scale across assays. The fungal native RNA-FM value comes from the historical Stage 3 record labeled `appendix`; the current reader accepts native unimodal records under either convolution label because they do not use the grid's transposed convolution. That selection rule does not establish complete historical producer identity. The grid control also shows why attributing the entire native-score difference to pretraining would be too strong.

**A stronger replacement is not the same finding as an added fusion benefit.** Fungal concatenation with mRNA-FM scores 0.853, compared with 0.793 for the original triple: the saved difference is +0.061 [0.034, 0.087]. But mRNA-FM alone scores 0.852, and the same fused model's gain over it is +0.001 [−0.014, 0.016]. The replacement improves this fitted pipeline without resolving an additional benefit from the other two streams.

mRNA-FM has higher native single-encoder scores on fungal expression, E. coli and stability, but lower scores on CoV-Vac. CoV-Vac's inputs are designed RNA fragments, not verified CDS; grouping them into triplets does not establish a biological reading frame. Its chemical-degradation target also differs from cellular expression and standardized stability. See the [target definitions](../../docs/methods/data-and-targets.md#cov-vac-chemical-degradation-of-designed-rna).

Across these four mRNA-FM replacement triples, none of the saved concatenation, gated or cross-attention intervals resolves a positive gain over the validation-selected single-encoder reference, which is mRNA-FM in each case. Some intervals include zero and some show a loss; an unresolved interval does not establish equivalence. These intervals resample test-sequence groups for one fitted seed. They do not measure training-seed variability or remove the known stability train/test overlap.

### 4. What remains unexplained, and how it informs fusion

The results are consistent with mRNA-FM making useful coding-sequence features accessible to the predictor. They do not identify whether the advantage comes from its corpus, codon grouping, capacity, wider representations, predictor dimensions or their combination. Likewise, a small fusion gain can reflect overlapping useful representations, an ineffective fusion head, the inherited tuning recipe or limited evidence. It does not show that the additional encoder contains no useful information.

This gives the next interpretation a concrete starting point: what target-relevant information could the other streams provide beyond a strong RNA representation? For example, a protein representation could expose amino-acid properties associated with the expression assay, while a nucleotide representation preserves synonymous variation. Both are views derived from the same supplied sequence here. Their biological relevance and incremental predictive value require evidence; simply naming them different modalities does not establish complementarity.

Comparable training-seed repeats would test whether the observed pattern persists. The existing RNA-FM grid control helps examine token resolution, and the paired comparisons distinguish improvement over the original pipeline from improvement over the selected single encoder. A subsequent combination should be motivated by a specific remaining information gap and a falsifiable comparison, including the possibility of no gain. This discussion proposes explanations to investigate; it does not select a new dataset or change the current experiment.

The following alignment section adds another view of representation relationships. Its measurements come from the Stage 1 stability sample, so agreement with this interpretation would remain sample-specific rather than explain every benchmark's outcome.

### Relate the comparisons to Stage 1 alignment

The Stage 2 decision is to report alignment beside fusion results. A replacement changes two of the triple's three encoder pairs. The table below gives Stage 1's raw and composition-controlled alignment for those pairs, using its 981-sequence mRNA stability sample, where all twelve encoders were measured together on coding sequences.

**Scope of the evidence.** These are measurements on that Stage 1 sample. They supply the representation measurements being examined under H2; they are not newly measured alignment on each fusion benchmark or an independently validated rule for predicting fusion gains.

**Open discrepancy.** The table reads the [Stage 1 capture](../../docs/figures/stability/results.json). Its raw CKA agrees with the saved Stage 1 notebook, but some composition-controlled values differ: RNA-FM with RiNALMo is 0.486 in the capture and 0.378 in the notebook. As recorded in the [run records](../../docs/run-records.md), that discrepancy remains unresolved. Treat the controlled values as provisional.

In [ ]:
def alignment_matrix(capture, name):
    """Restore encoder-labeled axes from the saved JSON matrix."""
    matrix_data = capture["similarity"][name]
    return pd.DataFrame(
        matrix_data["data"], index=matrix_data["index"], columns=matrix_data["columns"],
    )


def alignment_table(capture, configured_swaps):
    """Look up each replacement's Stage 1 similarity to retained and original encoders."""
    cka = alignment_matrix(capture, "Linear CKA")
    cka_controlled = alignment_matrix(capture, "Linear CKA after control")
    alignment = []
    for modality, encoder_key in sorted(configured_swaps):
        triple_keys = {**ORIGINAL, modality: encoder_key}
        # Read similarity to each of the two encoders retained in the triple.
        for other_modality in (slot for slot in ORIGINAL if slot != modality):
            retained_encoder = triple_keys[other_modality]
            alignment.append({
                "swap": ENCODERS[encoder_key].label,
                "pair": f"{ENCODERS[encoder_key].label} + {ENCODERS[retained_encoder].label}",
                "CKA": cka.loc[encoder_key, retained_encoder],
                "CKA after composition control": cka_controlled.loc[encoder_key, retained_encoder],
            })
        # Also compare the replacement with the original encoder it displaced.
        alignment.append({
            "swap": ENCODERS[encoder_key].label,
            "pair": (
                f"{ENCODERS[encoder_key].label} vs the "
                f"{ENCODERS[ORIGINAL[modality]].label} it replaces"
            ),
            "CKA": cka.loc[encoder_key, ORIGINAL[modality]],
            "CKA after composition control": cka_controlled.loc[encoder_key, ORIGINAL[modality]],
        })
    return pd.DataFrame(alignment)


In [24]:
alignment = []
try:
    # Reuse the Stage 1 capture; no alignment is recomputed for these benchmarks.
    stage1 = json.loads(STAGE1_ALIGNMENT.read_text())
    # A swap may appear on several benchmarks, but this shared Stage 1 lookup is needed once.
    configured_swaps = {
        swap for benchmark in BENCHMARKS for swap in swaps_for(benchmark)
    }
    alignment = alignment_table(stage1, configured_swaps)
    display(alignment.set_index(["swap", "pair"]).round(3))
except (OSError, ValueError, KeyError, TypeError) as exc:
    if not READ_SAVED_RESULTS:
        raise
    print(f"Stage 1 alignment unavailable: {exc}. No alignment was recalculated.")


CKA  CKA after composition control
swap              pair                                                                                    
HyenaDNA large    HyenaDNA large + RNA-FM                             0.398                          0.228
                  HyenaDNA large + ESM-2 8M                           0.161                          0.109
                  HyenaDNA large vs the NT v2 100M multi-species ...  0.736                          0.117
NT 500M human-ref NT 500M human-ref + RNA-FM                          0.530                          0.196
                  NT 500M human-ref + ESM-2 8M                        0.163                          0.126
                  NT 500M human-ref vs the NT v2 100M multi-speci...  0.398                          0.064
ESM-2 150M        ESM-2 150M + NT v2 100M multi-species               0.067                          0.030
                  ESM-2 150M + RNA-FM                                 0.200                          0.104
                  ESM-2 150M vs the ESM-2 8M it replaces              0.553                          0.391
ESM-2 35M         ESM-2 35M + NT v2 100M multi-species                0.083                          0.045
                  ESM-2 35M + RNA-FM                                  0.199                          0.175
                  ESM-2 35M vs the ESM-2 8M it replaces               0.744                          0.729
ProtBERT          ProtBERT + NT v2 100M multi-species                 0.058                          0.027
                  ProtBERT + RNA-FM                                   0.116                          0.083
                  ProtBERT vs the ESM-2 8M it replaces                0.199                          0.085
CaLM              CaLM + NT v2 100M multi-species                     0.183                          0.058
                  CaLM + ESM-2 8M                                     0.433                          0.421
                  CaLM vs the RNA-FM it replaces                      0.632                          0.194
mRNA-FM           mRNA-FM + NT v2 100M multi-species                  0.030                          0.044
                  mRNA-FM + ESM-2 8M                                  0.046                          0.106
                  mRNA-FM vs the RNA-FM it replaces                   0.126                          0.206
RiNALMo 150M      RiNALMo 150M + NT v2 100M multi-species             0.209                          0.089
                  RiNALMo 150M + ESM-2 8M                             0.339                          0.227
                  RiNALMo 150M vs the RNA-FM it replaces              0.809                          0.486

### What this investigation supports

Replacing an encoder can improve prediction under the retained BioLangFusion recipe, but improving the fused model and demonstrating an extra benefit from fusion are different outcomes. The fungal mRNA-FM comparison illustrates the first: much of the improvement is already available from the replacement alone. CoV-Vac with HyenaDNA cross-attention and E. coli with RiNALMo gated fusion supply positive paired differences against their declared single-encoder references, so the saved evidence does not support dismissing fusion across the board.

The differences do not isolate why a representation helps. Pretraining populations, token resolution, encoder capacity, the predictor's input dimensions, baseline selection, and inherited tuning remain possible contributors. Stage 1 similarity supplies a reason to ask about complementarity, but these tables do not establish a general relationship between alignment and fusion gain, particularly while the controlled-CKA discrepancy remains unresolved.

The retained examples above describe one training seed; the coverage tables distinguish those observations from any available repeats. Paired test-row intervals do not supply missing training repeats or remove overlap in the inherited stability split. The evidence therefore identifies specific comparisons worth repeating and interpreting, rather than a universal winning encoder or fusion method.

The contribution to the wider investigation is therefore a set of conditional observations: which encoder changes help, where a combination improves on the declared single reference, and which heads succeed under the same local recipe. Before selecting another combination or method, state what additional predictive pattern is expected, why the target and inputs make it plausible, and what outcome would contradict it. Training-seed repeats assess stability of the observed differences; a new dataset or held-out setting would test whether the expectation transfers. Neither every missing repeat nor every possible replacement is automatically a prerequisite for asking a focused question.

## Choices and limitations

These choices define the present comparison in addition to Phase A's settings.

| Choice | Reason | Interpretation limit |
| --- | --- | --- |
| One training seed per execution | Keeps manual seed repeats separate and reusable | Retained examples describe seed 0; a three-seed summary requires results for all requested seeds. Test-row bootstrap intervals do not measure training-seed variation |
| Reuse the original triple's gated-head entropy weight | Holds this tuning choice fixed across replacements | A replacement may behave differently at its own validation-selected weight |
| Leave DNABERT-2 outside the sweep | Variable-length byte-pair tokens need a fixed grid adapter | This DNA candidate remains untested here |
| Add RiNALMo and ESM-2 35M | Test replacements that Stage 1 found similar to the originals | Similarity motivates an H2 comparison; it does not guarantee little predictive change |
| Use alignment from the Stage 1 stability sample | Provides the existing twelve-encoder comparison | Values are sample-specific, and the controlled-CKA discrepancy remains open |
| Expand stability from the original three replacements to the full configured set | Compare the same replacement choices across benchmarks | Fit coverage and paired-interval coverage are separate; the retained three-replacement intervals do not describe the expanded set |
| Defer two-encoder replacements | Candidate triples depend on the single-swap evidence | This sweep does not complete the plan's later triple comparisons |